# DARA-ABR: counterfactual reward diagnosis for learning-based ABR

**Question.** Does the training reward of a learning-based ABR agent prefer actions that go badly under the intended
QoE, and if so, can the diagnosis set the fix automatically?

| Triage label | Meaning | Signal pattern |
|---|---|---|
| `OPT_FAILURE` | PPO is not optimising its own reward | low policy–value agreement **A** |
| `ENV_DEGENERATE` | one action is genuinely best in most states (no headroom) | high dominance **D**, low misalignment **M** |
| `REWARD_INDUCED` | the reward prefers an action that goes badly under the intended objective | high **D**, high **M** |
| `HEALTHY` | the preferred action changes with conditions | low **D** |

**One execution of this notebook = one job.** Parameters are injected with papermill (see `scripts/hpc/`); run it from
the repository root.

* `MODE='run'`: builds behaviour-cloning data from a rule-based expert, trains BC → PPO (optionally with DARA or DARA-L,
  or trains a Pensieve / COREL baseline), diagnoses the trained policy on a fixed set of 360 states with
  common-random-number counterfactual rollouts, and evaluates it on the shared test sessions. Output: `OUT_ROOT/RUN_NAME/`.
* `MODE='landscape'`: no training; maps the policy-independent signals (D, M, ΔQ, cap saturation, reward-best vs
  intended-best constant tier) over a grid of reward settings. Output: `OUT_ROOT/landscape/`.
* `MODE='generalize'`: plays a trained checkpoint, unchanged, on networks and chunk-size ladders it never saw.
  Output: `OUT_ROOT/generalize/gen_<run>.csv`.
* `MODE='aggregate'`: summary of a set of triage runs (not used for the paper figures).

**Environment.** 4-s chunks, 5 tiers, buffer ≤ 10 s. Network: the urban mmWave generator
(`src/dara_abr/urban_generator.py`, a six-state channel model calibrated on Lumos5G, eight scenarios) or per-second
throughput traces (`data/processed/*.npz`). Downloads integrate the time-varying throughput (`DL_MODEL='integrated'`).
Chunk sizes (`LADDER='content'`) come from a content-complexity generator calibrated on Puffer `video_size` data
(`data/processed/content_params.json`): size_b(t) = `BASE_CHUNK_MB`[b] × κ_b × m_b(t), with κ (title complexity) drawn
once per session and m(t) (scene complexity) following semi-Markov regimes; the whole session's sizes are fixed at
reset, so they never depend on the network.

**Legacy switches.** `UNITS_FIX`, `SIZE_CONSISTENT`, `DL_MODEL='window'` and `LADDER='relative'` reproduce an earlier
version of the environment and are kept only for comparison; every paper run uses `UNITS_FIX=1`, `DL_MODEL='integrated'`,
`LADDER='content'`.

**Requires** `src/dara_abr/urban_generator.py`, `data/processed/content_params.json`, and for trace networks
`data/processed/irish5g_download.npz` / `data/processed/lumos5g.npz` (built by `scripts/data/prepare_traces.py`).


In [ ]:
import os, sys, json, copy, time, random, glob, pickle, itertools, ast, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
try:
    from scipy import stats as sps
except ImportError:
    sps = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)

In [ ]:
# ── run identity ───────────────────────────────────────────────────────────────
RUN_NAME       = "example_run"
OUT_ROOT       = "runs/train"
# 'run' = train + diagnose one regime/seed; 'aggregate' = combine all runs; 'landscape' = reward-grid sweep, no training;
# 'timeline'   = replay test sessions of trained checkpoints chunk by chunk (TL_RUNS)
# 'generalize' = evaluate a trained checkpoint (GEN_CKPT) on unseen networks / chunk sizes, with and without the runtime guard
MODE           = "run"
REGIME         = "env_degenerate_rel"
# by construction: OPT_FAILURE / ENV_DEGENERATE / REWARD_INDUCED / HEALTHY / PROBE
EXPECTED_LABEL = "ENV_DEGENERATE"
SEED           = 42
# 1 = tiny sizes for an end-to-end check (minutes)
SMOKE          = 0

# ── environment ────────────────────────────────────────────────────────────────
# 'relative' (TIER_MULT), 'fixed' (BASE_CHUNK_MB) or 'content' (BASE_CHUNK_MB x Puffer-calibrated content generator)
LADDER          = "relative"
# 0 = legacy units (MB / Mbps, no x8); 1 = corrected (x8)
UNITS_FIX       = 0
# 0 = legacy behaviour (size redrawn in step); 1 = download the observed size
SIZE_CONSISTENT = 0
# download model: 'integrated' = bits arrive along the time-varying trace and the trace clock advances by the real
# elapsed time (Pensieve-style); 'window' = legacy behaviour (mean of a fixed 4-s slot held for the whole download)
DL_MODEL        = "integrated"
TIER_MULT       = [0.30, 0.55, 0.85, 1.25, 1.90]
BASE_CHUNK_MB   = [1.5, 3.0, 6.0, 12.0, 22.0]
# LADDER='content' only: generator parameters calibrated on Puffer video_size data
CONTENT_MODEL     = "data/processed/content_params.json"
# cap on the scene multiplier m (Puffer 99th percentile = 3.33)
CONTENT_CAP       = 3.33
# 1 = include the slow AR(1) component (memory over 20 s - 2 min)
CONTENT_SLOW      = 1
# scales the log-multiplier deviations (robustness studies only; 1.0 = as calibrated)
CONTENT_VAR_SCALE = 1.0
MAX_BUF         = 10.0
# network source: 'generator' (urban mmWave generator) or 'trace' (per-second throughput traces in TRACE_FILE)
NET_SOURCE      = "generator"
# generator only: restrict to these scenarios (empty = the training mix); multiply throughput by NET_SCALE
NET_SCENARIOS   = []
NET_SCALE       = 1.0
# trace only: npz of per-second throughput (Mbit/s); keys starting with TRACE_PREFIX are used ("" = all)
TRACE_FILE      = "data/processed/irish5g_download.npz"
TRACE_PREFIX    = ""
# trace only: share of the selected traces held out for testing (0 = none; stratified by name prefix, seeded);
# held-out traces are used only by the final test sessions, never for training, diagnosis or validation
TRACE_HOLDOUT_FRAC = 0.0
TRACE_HOLDOUT_SEED = 7
N_BITRATES      = 5
K               = 8
N_BUF_CATS      = 10

# ── training reward (the reward under diagnosis) ───────────────────────────────
QUALITY   = [1.0, 1.92, 2.61, 3.30, 4.00]
# >0 replaces QUALITY with 4*(1-exp(-K*TIER_MULT))
QUALITY_K = 0.0
ALPHA     = 4.3
BETA      = 1.0
# 1e6 = uncapped
RN_CAP    = 5.0
CLIP_LO   = -60.0
# freeze price rho: multiplies the training reward's rebuffering penalty (1.0 = original reward)
REBUF_SCALE = 1.0
CLIP_HI   = 7.0

# ── reference objective: fixed, reward-independent (used for M and ground truth) ─
REF_QUALITY = [1.0, 1.92, 2.61, 3.30, 4.00]
# per second of rebuffering (= ALPHA / CHUNK_DUR), uncapped, unclipped
REF_ALPHA   = 1.075
# M is also reported at these weights (sensitivity)
REF_ALPHA_GRID = [0.5, 1.075, 2.15, 4.3]
REF_BETA    = 1.0

# ── behaviour cloning ──────────────────────────────────────────────────────────
BC_EPISODES        = 600
BC_EPOCHS          = 8
BC_LR              = 3e-4
BC_BATCH           = 256
BC_WD              = 1e-4
BC_DROPOUT         = 0.15
BC_LABEL_SMOOTHING = 0.10
EXPERT_SAFETY      = 0.9
EXPERT_BUF_LO      = 2.0
EXPERT_BUF_HI      = 8.0
EXPERT_CV_HIGH     = 0.3
# expert / bounded-random / conservative
LABEL_MIX          = [0.6, 0.2, 0.2]

# ── PPO ────────────────────────────────────────────────────────────────────────
PPO_EPISODES     = 10000
PPO_ROLLOUT      = 512
PPO_EPOCHS       = 4
PPO_BATCH        = 64
PPO_LR           = 1e-4
PPO_GAMMA        = 0.99
PPO_LAMBDA       = 0.95
PPO_CLIP         = 0.2
PPO_C1           = 0.5
PPO_C2           = 0.05
PPO_MAX_GRAD     = 0.5
# 'running' (fixed) or 'batch' (legacy)
RETURN_NORM_MODE = "running"
# 1 (fixed) or 0 (legacy)
VALUE_DENORM     = 1
EMBED_DIM, ENC_OUT, FC_APP_DIM, LSTM_LAYERS = 16, 32, 128, 1

# ── counterfactual diagnostic ──────────────────────────────────────────────────
# fixed: identical diagnostic states for every run on the same environment
DIAG_SEED        = 2026
N_DIAG           = 360
POOL_FACTOR      = 4
SNAP_PROB        = 0.3
BEHAV_RANDOM     = 0.3
H_LIST           = [5, 10]
H_MAIN           = 10
N_REP            = 4
DIAG_GAMMA       = 0.99
# tolerance for A, as a fraction of each state's Q_follow range (max - min)
AGREE_TOL        = 0.10
# reference-return margin for M
REF_MARGIN       = 0.5
# seconds, for M_rebuf
REBUF_MARGIN     = 0.5
SCAS_MIN_STATES  = 5
# condition cells for SCAS (subset of level, vol, buf, trend)
SCAS_GROUP       = "level,buf"
N_BOOT           = 500

# ── triage thresholds ──────────────────────────────────────────────────────────
A_MIN  = 0.6
D_HIGH = 0.8
M_HIGH = 0.2

# ── independent ground truth ───────────────────────────────────────────────────
GT_EPISODES    = 60
EVAL_SEED      = 99
GT_FAIL_MARGIN = 0.05
GT_DOMINANT    = 0.9
MPC_LOOKAHEAD  = 5

# ── DARA and baselines (MODE='run') ──────────────────────────────────────────
# 'none' | 'preflight' (choose the cap before training) | 'online' (diagnose during PPO, relax the cap)
# | 'k' (original proposal: dominance-targeting curvature controller)
# | 'preflight_price' / 'online_price' (as preflight / online, then also set the freeze price rho, see PRICE_GRID)
# | 'preflight_lag' / 'online_lag' (DARA-L: when DARA acts it switches on PPO-Lagrangian with the freeze budget
#   LAG_TARGET set from the diagnosis = mean freeze per chunk of the intended objective's best hold action)
DARA_MODE      = "none"
CAP_GRID       = [0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 1e6]
# freeze-price grid for the *_price modes: when DARA acts, after the cap step it picks the rho with the lowest
# intended-return regret of the reward-preferred actions (ties -> smaller rho)
PRICE_GRID     = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0, 4.0]
# PPO episodes between diagnoses (online / k)
DIAG_EVERY     = 1000
# consecutive actionable diagnoses before the cap is relaxed (online)
PERSIST        = 2
# episodes per policy for the on-policy gate during training
DIAG_EPISODES  = 20
# critic-only PPO updates after a surrogate change
WARMUP_UPDATES = 10
# detection thresholds (fixed before any paper run)
GATE_MIN       = -0.10
# DARA-k controller: E = W_D (D - D_TARGET) + W_Q max(0, dQ - DQ_TARGET); K <- clip(K + ETA_K sign(dominant tier - 2) E, 0, K_MAX)
D_TARGET, DQ_TARGET, W_D, W_Q, ETA_K, K_MAX = 0.8, 0.1, 1.0, 1.0, 10.0, 12.0
# adaptive entropy baseline: log c2 <- log c2 + ENT_LR (ENT_TARGET - entropy)
ENT_ADAPT, ENT_TARGET, ENT_LR = 0, 1.0, 0.05
# PPO-Lagrangian baseline: r - lambda * rebuffer_s ; lambda <- max(0, lambda + LAG_LR (mean rebuffer per chunk - LAG_TARGET))
LAGRANGE, LAG_TARGET, LAG_LR = 0, 0.5, 0.05

# ── benchmark algorithms (MODE='run') ─────────────────────────────────────────
# 'dara'     = this pipeline: BC-initialised actor-critic + PPO (DARA_MODE / LAGRANGE apply)
# 'pensieve' = Pensieve architecture (Mao et al., SIGCOMM 2017: scalars -> dense 128, histories and next sizes ->
#              1-D conv with 128 filters of size 4, merge -> dense 128; separate actor and critic networks),
#              trained from scratch (no BC) with PPO, the widely used Pensieve-PPO variant of its A3C training
# 'corel'    = COREL-style constrained RL (Hu et al., IEEE CQR 2023, re-implemented from the paper's description):
#              Pensieve architecture plus a separate stall critic; primal-dual PPO that maximises the QoE reward
#              subject to mean stall per chunk <= COREL_TARGET; the dual variable is updated from the measured stall
ALGO           = "dara"
# training reward: 'surrogate' = the reward under diagnosis (cap, clip, shaping); 'ref' = the intended per-chunk QoE u_t
REWARD_MODE    = "surrogate"
COREL_TARGET, COREL_LR = 0.5, 0.05
# official-recipe baselines (A3C as in the released COREL / Pensieve code; PyTorch re-implementation):
# 'pensieve_official' : Pensieve network, reward (bitrate_Mbps - COREL_MU0 x stall_s - |d bitrate_Mbps|) / A3C_STATE_SCALE, mu fixed
# 'corel_official'    : COREL network (Pensieve + 4 future-throughput inputs, true trace +/- 16% noise at offsets 0,1,3,6 s),
#                       same reward with a dual variable mu: every COREL_MU_EVERY epochs mu <- max(mu + COREL_MU_LR x
#                       (validation stall of the best checkpoint - COREL_EPS), 0); stall ("penalty") critic trained alongside
# tier bitrates are the nominal ladder bitrates BASE_CHUNK_MB x 8 / 4 s (x4 ladder: 12-176 Mbps, the range of COREL's 1.5-152 Mbps)
COREL_MU0, COREL_EPS, COREL_MU_LR, COREL_MU_EVERY = 160.0, 0.01, 0.1, 100
A3C_AGENTS, A3C_SEQ, A3C_ACTOR_LR, A3C_CRITIC_LR, A3C_STATE_SCALE = 16, 100, 1e-4, 1e-3, 100.0
# entropy weight stages (COREL's documented sweep), each used for an equal share of the PPO_EPISODES budget
A3C_ENTROPY     = [2.0, 1.0, 0.5, 0.25, 0.1, 0.05, 0.01]
A3C_VALID_EVERY, A3C_VALID_EPISODES, A3C_PATIENCE = 100, 20, 200

# ── landscape sweep (MODE='landscape' only; no training) ───────────────────────
LS_RN_CAP       = [0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 1e6]
LS_QUALITY_K    = [0.0, 3.0, 5.0, 8.0, 12.0]
# extra configs at RN_CAP=1e6, K_Q=0 with the top-tier quality raised to these values
LS_QTOP         = [6.0, 8.0, 12.0]
# multiplies BASE_CHUNK_MB (fixed and content ladders; ignored for the relative ladder)
LS_LADDER_SCALE = [0.25, 1.0, 2.0, 4.0, 8.0]
# extra configs at K_Q=0 for every LS_RN_CAP with the freeze price rho set to these values (1.0 = no extra configs)
LS_REBUF_SCALE  = [1.0]
LS_BOOT         = 200

# ── generalisation test (MODE='generalize'; no training) ──────────────────────
# run directory holding checkpoint.pt; "" or "none" = reference job (always-one-tier policies and RobustMPC only)
GEN_CKPT        = ""
# available: seen, lumos5g, lumos5g_walking, lumos5g_driving, irish_static, irish_driving, gen_harsh, gen_half
GEN_NETWORKS    = ["seen", "lumos5g", "irish_static", "irish_driving"]
# available: x4, x1, x8, x4_bursty
GEN_CONTENT     = ["x4", "x1", "x8"]
GEN_EPISODES    = 100
GEN_SEED        = 99
# per-second throughput traces of the test networks (scripts/data/prepare_traces.py)
IRISH_TRACE_FILE = "data/processed/irish5g_download.npz"
LUMOS_TRACE_FILE = "data/processed/lumos5g.npz"
# runtime guard variants: 'none' (bare policy), 'price', 'fallback', 'both'
RT_MODES        = ["none"]

# ── session timeline (MODE='timeline'; no training) ───────────────────────────
# replays the test sessions of trained checkpoints chunk by chunk and writes <OUT_ROOT>/timeline/timeline_<TL_TAG>.csv
# TL_RUNS: comma-separated run folders (or names under runs/train); the word "robustmpc" adds RobustMPC on the same sessions
TL_RUNS         = ""
TL_TAG          = "timeline"
# freeze budget, s per chunk; < 0 = derive it with the DARA-L rule on the training environment
RT_BUDGET       = -1.0
RT_WINDOW       = 8
RT_ETA          = 0.2
RT_LAMBDA_MAX   = 10.0
RT_OOD_QUANT    = 0.99
RT_OOD_PATIENCE = 3


In [ ]:
# ── coerce papermill string injections, apply SMOKE, build CFG ────────────────
for _n in ["A3C_ENTROPY", "TIER_MULT","BASE_CHUNK_MB","QUALITY","REF_QUALITY","LABEL_MIX","H_LIST","REF_ALPHA_GRID","CAP_GRID",
           "LS_RN_CAP","LS_QUALITY_K","LS_QTOP","LS_LADDER_SCALE","PRICE_GRID","LS_REBUF_SCALE",
           "NET_SCENARIOS","GEN_NETWORKS","GEN_CONTENT","RT_MODES"]:
    if isinstance(globals()[_n], str):
        globals()[_n] = ast.literal_eval(globals()[_n])
if str(TRACE_PREFIX).lower() == "none":          # papermill cannot pass an empty string
    TRACE_PREFIX = ""
if SMOKE:
    BC_EPISODES, BC_EPOCHS, PPO_EPISODES = 20, 1, 40
    DIAG_EVERY, DIAG_EPISODES, WARMUP_UPDATES, PERSIST = 10, 3, 1, 1
    N_DIAG, N_REP, H_LIST, H_MAIN, GT_EPISODES, N_BOOT = 40, 2, [5], 5, 5, 50
    A3C_AGENTS, A3C_VALID_EPISODES, A3C_VALID_EVERY, COREL_MU_EVERY = 4, 3, 2, 2
QUALITY_BASE = [float(x) for x in QUALITY]
if QUALITY_K > 0:
    QUALITY = list(4.0 * (1.0 - np.exp(-QUALITY_K * np.array(TIER_MULT))))
assert MODE in ("run", "aggregate", "landscape", "generalize", "timeline"), MODE
assert NET_SOURCE in ("generator", "trace"), NET_SOURCE
assert DL_MODEL in ("integrated", "window"), DL_MODEL
assert DARA_MODE in ("none", "preflight", "online", "k", "preflight_price", "online_price", "preflight_lag", "online_lag"), DARA_MODE
PRICE_GRID = sorted(float(p) for p in PRICE_GRID)
CAP_GRID = sorted(float(c) for c in CAP_GRID)
assert H_MAIN in H_LIST, "H_MAIN must be one of H_LIST"
assert LADDER in ("relative", "fixed", "content"), LADDER
assert ALGO in ("dara", "pensieve", "corel", "pensieve_official", "corel_official"), ALGO
assert REWARD_MODE in ("surrogate", "ref"), REWARD_MODE
if ALGO != "dara":
    assert DARA_MODE == "none" and not LAGRANGE, "Pensieve / COREL baselines run without DARA and without the PPO-Lagrangian"
    BC_EPOCHS = 0                                    # trained from scratch; BC rollouts still supply the input normalisation

class CFG: pass
C = CFG()
for _n, _v in list(globals().items()):
    if _n.isupper() and not _n.startswith("_") and isinstance(_v, (int, float, str, list)):
        setattr(C, _n, _v)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
RUN_DIR = Path(OUT_ROOT) / RUN_NAME
if MODE == "run":
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    json.dump({k: v for k, v in vars(C).items()}, open(RUN_DIR / "config.json", "w"), indent=2)
print("RUN_DIR:", RUN_DIR, "| QUALITY:", np.round(QUALITY, 3).tolist())

## 1. Environment

The training reward (progress-weighted α, capped freeze penalty, switch, cliff, oscillation, capped startup term,
clip) is the reward under diagnosis. `step()` additionally returns an `info` dict with raw freezing, cap/clip
saturation and the **intended return** (fixed objective, uncapped), which the diagnosis needs.


In [ ]:
if MODE in ("run", "landscape", "generalize", "timeline"):
    sys.path.insert(0, "src")
    from dara_abr.urban_generator import generate_session, SCENARIO_MIX

_TRACE_CACHE = {}
def load_traces(path):
    if path not in _TRACE_CACHE:
        if not Path(path).exists():
            raise FileNotFoundError(f"NET_SOURCE='trace' needs {path} next to the notebook")
        z = np.load(path); _TRACE_CACHE[path] = {k: np.asarray(z[k], dtype=float) for k in z.files}
    return _TRACE_CACHE[path]

# ── content-complexity chunk-size generator (LADDER='content') ─────────────────
# size_b(t) = BASE_CHUNK_MB[b] * kappa_b * m_b(t)   (independent of the network, like a real server's manifest)
#   kappa : title complexity, one draw per session from the empirical Puffer distribution; tier exponent gamma_kappa_b
#   m(t)  : scene complexity: semi-Markov regimes (empirical dwell times, jump matrix) with AR(1) wobble inside a
#           regime, plus an optional slow AR(1) component; shared across tiers with exponent gamma_m_b and a small
#           tier-specific deviation sigma_idio_b; capped at CONTENT_CAP
#   kappa_b and m_b are normalised to mean 1, so BASE_CHUNK_MB stays each tier's average chunk size.
_CONTENT_CACHE = {}

def _content_logm(cm, n, rng):
    mu, sd, rho = cm["mu"], cm["sd"], cm["rho"]
    K = len(mu)
    x = np.empty(n)
    k = int(rng.choice(K, p=cm["share"]))
    t, first = 0, True
    while t < n:
        d = 1 + int(np.searchsorted(cm["dwell_cdf"][k], rng.random(), side="right"))
        if first:                                   # stationary start: residual life of the first regime
            d, first = 1 + int(rng.integers(d)), False
        e = mu[k] + sd[k] * rng.standard_normal()
        for j in range(min(d, n - t)):
            if j:
                e = mu[k] + rho[k] * (e - mu[k]) + sd[k] * np.sqrt(1.0 - rho[k] ** 2) * rng.standard_normal()
            x[t] = e
            t += 1
        k = int(rng.choice(K, p=cm["jump"][k]))
    if cm["slow_w"] > 0:
        phi, s = cm["slow_phi"], np.empty(n)
        s[0] = rng.standard_normal()
        z = rng.standard_normal(n)
        for i in range(1, n):
            s[i] = phi * s[i - 1] + np.sqrt(1.0 - phi ** 2) * z[i]
        x = cm["mh"] + np.sqrt(1.0 - cm["slow_w"]) * (x - cm["mh"]) + np.sqrt(cm["slow_w"]) * cm["sh"] * s
    if cm["var_scale"] != 1.0:
        x = cm["mh"] + cm["var_scale"] * (x - cm["mh"])
    return x

def _content_raw(cm, n, rng):
    # unnormalised (n, tiers) scene multipliers and per-tier title multipliers
    lk = float(np.interp(rng.random(), cm["kq_grid"], cm["log_kappa_q"]))
    kap = np.exp(cm["gamma_kappa"] * lk)
    h = _content_logm(cm, n, rng)
    eps = rng.standard_normal((n, len(cm["gamma_m"]))) * cm["sigma_idio"]   # tier-specific deviation (corr < 1)
    m = np.minimum(np.exp(np.outer(h, cm["gamma_m"]) + eps), cm["cap"])
    return kap, m

def load_content_model(cfg):
    key = (str(cfg.CONTENT_MODEL), float(cfg.CONTENT_CAP), int(cfg.CONTENT_SLOW), float(cfg.CONTENT_VAR_SCALE))
    if key in _CONTENT_CACHE:
        return _CONTENT_CACHE[key]
    if not Path(cfg.CONTENT_MODEL).exists():
        raise FileNotFoundError(f"LADDER='content' needs {cfg.CONTENT_MODEL} next to the notebook")
    P = json.load(open(cfg.CONTENT_MODEL))
    R = P["regimes"]
    share = np.asarray(R["share"], float); share /= share.sum()
    mu, sd = np.asarray(R["mu"], float), np.asarray(R["sd"], float)
    mh = float(np.sum(share * mu)); sh = float(np.sqrt(np.sum(share * (sd ** 2 + mu ** 2)) - mh ** 2))
    jump = np.asarray(R["jump"], float); jump /= jump.sum(1, keepdims=True)
    cm = dict(mu=mu, sd=sd, rho=np.asarray(R["rho"], float), share=share, jump=jump,
              dwell_cdf=[np.cumsum(p) / np.sum(p) for p in R["dwell_pmf"]], mh=mh, sh=sh,
              slow_w=float(P["slow"]["w"]) if cfg.CONTENT_SLOW else 0.0, slow_phi=float(P["slow"]["phi"]),
              var_scale=float(cfg.CONTENT_VAR_SCALE), cap=float(cfg.CONTENT_CAP),
              gamma_m=np.asarray(P["tiers"]["gamma_m"], float), sigma_idio=np.asarray(P["tiers"]["sigma_idio"], float),
              gamma_kappa=np.asarray(P["tiers"]["gamma_kappa"], float),
              log_kappa_q=np.asarray(P["kappa"]["log_quantiles"], float),
              kq_grid=np.linspace(0.01, 0.99, len(P["kappa"]["log_quantiles"])))
    assert len(cm["gamma_m"]) == cfg.N_BITRATES, "content model tiers must match N_BITRATES"
    # normalisers (fixed seed, deterministic): E[kappa_b] over the quantile grid, E[m_b] by simulation
    cm["z_kappa"] = np.exp(np.outer(cm["log_kappa_q"], cm["gamma_kappa"])).mean(0)
    sim = np.random.default_rng(20261008)
    ms = [_content_raw(cm, 1000, sim)[1] for _ in range(200)]
    cm["z_m"] = np.concatenate(ms).mean(0)
    cm["md5"] = hashlib.md5(open(cfg.CONTENT_MODEL, "rb").read()).hexdigest()[:8]
    _CONTENT_CACHE[key] = cm
    return cm

def content_session(cm, n, rng):
    # (n, tiers) multipliers kappa_b * m_b(t); mean 1 per tier
    kap, m = _content_raw(cm, n, rng)
    return (kap / cm["z_kappa"])[None, :] * (m / cm["z_m"][None, :])


def normalise_obs(raw, norm):
    # raw -> policy inputs, z-normalised with the BC-rollout statistics.
    n = norm or dict(remain_mean=0., remain_std=1., size_mean=0., size_std=1.,
                     dl_mean=0., dl_std=1., tp_mean=0., tp_std=1.)
    return dict(
        remain=float(np.clip((raw["remain"] - n["remain_mean"]) / (n["remain_std"] + 1e-8), -5, 5)),
        next_sizes=((np.asarray(raw["sizes"]) - n["size_mean"]) / (n["size_std"] + 1e-8)).astype(np.float32),
        dl_times=((np.asarray(raw["dl"]) - n["dl_mean"]) / (n["dl_std"] + 1e-8)).astype(np.float32),
        throughputs=((np.asarray(raw["tp"]) - n["tp_mean"]) / (n["tp_std"] + 1e-8)).astype(np.float32),
    )

def holdout_traces(keys, cfg):
    """Deterministic held-out subset of `keys` (sorted names): within each name prefix, round(frac * n) traces
    (at least one when frac > 0), chosen with TRACE_HOLDOUT_SEED."""
    frac = float(getattr(cfg, "TRACE_HOLDOUT_FRAC", 0.0) or 0.0)
    if frac <= 0:
        return []
    groups = {}
    for k in sorted(keys):
        groups.setdefault(k.split("_")[0], []).append(k)
    out = []
    for gi, (g, ks) in enumerate(sorted(groups.items())):
        n = max(1, int(round(frac * len(ks))))
        assert n < len(ks), f"holding out {n} of {len(ks)} traces with prefix {g!r} leaves none for training"
        rng = np.random.default_rng([int(getattr(cfg, "TRACE_HOLDOUT_SEED", 7)), gi])
        out += [ks[j] for j in sorted(rng.choice(len(ks), size=n, replace=False))]
    return sorted(out)

class ABREnv:
    CHUNK_DUR = 4.0
    K = 8

    def __init__(self, cfg, norm=None, split="train", seed=0):
        self.cfg, self.norm = cfg, norm
        self.MAX_BUF = cfg.MAX_BUF
        self.unit = 8.0 if cfg.UNITS_FIX else 1.0
        base = seed if split == "train" else seed + 1_000_000
        self._rng = np.random.default_rng(base)
        self._frng = np.random.default_rng(base + 7_000_007)   # COREL's prediction noise only (never touches _rng)
        if cfg.NET_SOURCE == "trace":
            tr = load_traces(cfg.TRACE_FILE)
            keys = sorted(k for k in tr if k.startswith(cfg.TRACE_PREFIX))
            assert keys, f"no traces start with {cfg.TRACE_PREFIX!r} in {cfg.TRACE_FILE}"
            hold = holdout_traces(keys, cfg)
            if hold:                                     # 'test' = held-out traces only; everything else = the rest
                keys = hold if split == "test" else [k for k in keys if k not in set(hold)]
            self._traces = {k: tr[k] for k in keys}
            self._names = keys
            self._probs = np.full(len(self._names), 1.0 / len(self._names))
        elif cfg.NET_SCENARIOS:
            self._names = list(cfg.NET_SCENARIOS)
            p = np.array([SCENARIO_MIX[n] for n in self._names], dtype=float)
            self._probs = p / p.sum()
        else:
            self._names = list(SCENARIO_MIX)
            p = np.array(list(SCENARIO_MIX.values()), dtype=float)
            self._probs = p / p.sum()
        self.content = load_content_model(cfg) if cfg.LADDER == "content" else None
        self.reset()

    def reset(self):
        scenario = str(self._rng.choice(self._names, p=self._probs))
        dur = int(self._rng.integers(250, 900))
        if self.cfg.NET_SOURCE == "trace":
            tr = self._traces[scenario]
            off = int(self._rng.integers(0, len(tr)))
            thr = np.take(tr, np.arange(off, off + dur), mode="wrap")   # a random window of a real trace (wraps)
        else:
            thr = generate_session(scenario, dur, self._rng)["throughput"]
        if self.cfg.NET_SCALE != 1.0:
            thr = thr * self.cfg.NET_SCALE
        self.scenario = scenario
        self.tp_trace = np.maximum(thr, 0.1)
        N = len(self.tp_trace)
        max_possible = (N - self.K) // int(self.CHUNK_DUR)
        self.n_chunks = int(min(self._rng.integers(20, 155), max_possible))
        self.buffer = float(self._rng.uniform(4.0, 12.0))
        self.last_br = int(self._rng.integers(0, self.cfg.N_BITRATES))
        # content ladder: the whole session's chunk-size multipliers are fixed at reset (the manifest)
        self.manifest = (content_session(self.content, self.n_chunks + 1, self._rng)
                         if self.content is not None else None)
        self.t = 0
        self.clock = float(self.K)                       # trace time in seconds (integrated download model)
        self.dl_hist = [self.CHUNK_DUR] * self.K
        self.tp_hist = list(self.tp_trace[:self.K].astype(float))
        self.action_hist = []
        self.startup_done = False
        self.ep_reward = self.ep_reward_raw = self.ep_rebuffer = self.ep_ref = 0.0
        self._obs_sizes = None
        return self._state()

    def _categorize(self, buf):
        return int(np.clip(np.searchsorted(np.linspace(0, self.MAX_BUF, 11), buf, side="right") - 1, 0, 9))

    def _draw_sizes(self):
        if self.cfg.LADDER == "content":
            # sizes of the next chunk from the manifest: a property of the video, independent of the network
            row = self.manifest[min(self.t, len(self.manifest) - 1)]
            return [self.cfg.BASE_CHUNK_MB[b] * float(row[b]) for b in range(self.cfg.N_BITRATES)]
        # one draw of N_BITRATES uniforms per call, as in _state()
        mult = 0.70 + 0.60 * self._rng.random(self.cfg.N_BITRATES)
        if self.cfg.LADDER == "relative":
            recent = np.asarray(self.tp_hist[-self.K:], dtype=np.float64)
            harm = self.K / np.sum(1.0 / np.maximum(recent, 0.1))
            cv = float(np.std(recent) / (np.mean(recent) + 1e-8))
            safe = harm / (1.0 + cv)
            return [self.cfg.TIER_MULT[b] * safe * self.CHUNK_DUR / 8.0 * mult[b] for b in range(self.cfg.N_BITRATES)]
        return [self.cfg.BASE_CHUNK_MB[b] * mult[b] for b in range(self.cfg.N_BITRATES)]

    def future_sizes(self, h):
        # (h, tiers) sizes of the next h chunks from the manifest (content ladder only)
        rows = self.manifest[np.minimum(np.arange(self.t, self.t + h), len(self.manifest) - 1)]
        return rows * np.asarray(self.cfg.BASE_CHUNK_MB, dtype=float)[None, :]

    def _state(self):
        ns = self._draw_sizes()
        self._obs_sizes = ns
        raw = dict(remain=float(self.n_chunks - self.t), sizes=np.array(ns),
                   dl=np.array(self.dl_hist[-self.K:]), tp=np.array(self.tp_hist[-self.K:]))
        obs = normalise_obs(raw, self.norm)
        if getattr(self.cfg, "ALGO", "dara") == "corel_official":
            # as in COREL's simulator: true throughput at offsets 0, 1, 3, 6 s ahead, each +/- uniform(-72, 72)/439 noise
            fr = getattr(self, "_frng", None)
            if fr is None:
                fr = self._frng = np.random.default_rng(12345)
            n_ = len(self.tp_trace); c0 = int(self.clock)
            fut = np.array([self.tp_trace[(c0 + o) % n_] for o in (0, 1, 3, 6)], dtype=float)
            fut = np.maximum(0.0, fut + fr.integers(-72, 72, size=4) / 439.0 * fut)
            nn_ = self.norm or dict(tp_mean=0.0, tp_std=1.0)
            obs["future_tp"] = ((fut - nn_["tp_mean"]) / (nn_["tp_std"] + 1e-8)).astype(np.float32)
        obs.update(last_bitrate=self.last_br, buffer_cat=self._categorize(self.buffer), buffer_s=float(self.buffer),
                   _next_raw=ns, _raw=raw)
        return obs

    def _integrated_download(self, mbit):
        # walk the per-second trace from the current clock until `mbit` megabits have arrived (trace wraps)
        n, t, el = len(self.tp_trace), self.clock, 0.0
        while mbit > 1e-12:
            sec = int(t)
            rate = max(float(self.tp_trace[sec % n]), 0.1)
            frac = 1.0 - (t - sec)
            if rate * frac >= mbit:
                dt = mbit / rate; el += dt; t += dt; mbit = 0.0
            else:
                mbit -= rate * frac; el += frac; t += frac
        self.clock = t
        return el

    def step(self, action):
        drawn = self._draw_sizes()                        # legacy: step() re-calls _state() -> fresh draw
        ns = self._obs_sizes if (self.cfg.SIZE_CONSISTENT and self._obs_sizes is not None) else drawn
        if self.cfg.DL_MODEL == "window":
            ti = self.K + self.t * int(self.CHUNK_DUR)
            tp = max(float(np.mean(self.tp_trace[ti: ti + int(self.CHUNK_DUR)])), 0.1)
            dl_time = ns[action] * self.unit / tp
        else:
            dl_time = self._integrated_download(ns[action] * self.unit)
            tp = ns[action] * self.unit / max(dl_time, 1e-9)   # throughput as the player observes it
        rebuffer = max(0.0, dl_time - self.buffer)
        buf_after = self.buffer - min(self.buffer, dl_time) + self.CHUNK_DUR
        if self.cfg.DL_MODEL == "integrated" and buf_after > self.MAX_BUF:
            self.clock += buf_after - self.MAX_BUF          # player idles while the buffer is full
        self.buffer = min(buf_after, self.MAX_BUF)

        Q = self.cfg.QUALITY
        rn_raw = rebuffer / self.CHUNK_DUR
        rn = min(rn_raw, self.cfg.RN_CAP)
        prog = 1.0 - (self.t / max(self.n_chunks, 1))
        aeff = self.cfg.ALPHA * (0.4 + 0.6 * prog)
        sw = abs(Q[action] - Q[self.last_br])
        cliff = -0.4 * np.exp(-self.buffer / 3.0) if self.buffer < 5.0 else 0.0
        self.action_hist.append(action)
        osc = (-0.3 * float(np.var([Q[a] for a in self.action_hist[-4:]]))
               if len(self.action_hist) >= 4 else 0.0)
        start = -2.0 * min(dl_time / self.CHUNK_DUR, self.cfg.RN_CAP) if not self.startup_done else 0.0
        self.startup_done = True
        raw_reward = Q[action] - aeff * self.cfg.REBUF_SCALE * rn - self.cfg.BETA * sw + cliff + osc + start
        reward = float(np.clip(raw_reward, self.cfg.CLIP_LO, self.cfg.CLIP_HI))

        R = self.cfg.REF_QUALITY
        ref = R[action] - self.cfg.REF_ALPHA * rebuffer - self.cfg.REF_BETA * abs(R[action] - R[self.last_br])
        info = dict(rebuffer=rebuffer, dl_time=dl_time, ref=ref, raw_reward=raw_reward,
                    ref_q=R[action], ref_sw=abs(R[action] - R[self.last_br]),
                    capped=float(rn_raw > self.cfg.RN_CAP), clipped=float(raw_reward < self.cfg.CLIP_LO))

        self.dl_hist.append(dl_time)
        self.tp_hist.append(tp)
        self.last_br = action
        self.t += 1
        self.ep_reward += reward
        self.ep_reward_raw += raw_reward
        self.ep_rebuffer += rebuffer
        self.ep_ref += ref
        done = self.t >= self.n_chunks
        return (self._state() if not done else None), reward, done, info

def obs_to_tensors(obs_list, device=DEVICE):
    return (
        torch.as_tensor([o["last_bitrate"] for o in obs_list], dtype=torch.long, device=device),
        torch.as_tensor([o["buffer_cat"] for o in obs_list], dtype=torch.long, device=device),
        torch.as_tensor([[o["remain"]] for o in obs_list], dtype=torch.float32, device=device),
        torch.as_tensor(np.stack([o["next_sizes"] for o in obs_list]), dtype=torch.float32, device=device),
        torch.as_tensor(np.stack([o["dl_times"] for o in obs_list]), dtype=torch.float32, device=device),
        torch.as_tensor(np.stack([o["throughputs"] for o in obs_list]), dtype=torch.float32, device=device),
        # buffer in seconds / 10 (Pensieve's normalisation); only the Pensieve-architecture models read it
        torch.as_tensor([[o.get("buffer_s", o["buffer_cat"] + 0.5) / 10.0] for o in obs_list], dtype=torch.float32, device=device),
        # COREL's future-throughput input (zeros for every other model, which ignore it)
        torch.as_tensor(np.stack([o.get("future_tp", np.zeros(4, np.float32)) for o in obs_list]), dtype=torch.float32, device=device),
    )
print("environment defined | unit factor =", 8.0 if UNITS_FIX else 1.0)
if MODE == "run" and NET_SOURCE == "trace" and float(TRACE_HOLDOUT_FRAC) > 0:
    _keys = sorted(k for k in load_traces(TRACE_FILE) if k.startswith(TRACE_PREFIX))
    _hold = holdout_traces(_keys, C)
    json.dump(dict(trace_file=TRACE_FILE, prefix=TRACE_PREFIX, frac=float(TRACE_HOLDOUT_FRAC), seed=int(TRACE_HOLDOUT_SEED),
                   held_out=_hold, training=[k for k in _keys if k not in set(_hold)]),
              open(RUN_DIR / "holdout_traces.json", "w"), indent=1)
    print(f"held-out test traces ({len(_hold)} of {len(_keys)}):", _hold)

## 2. Rule-based expert and RobustMPC

The expert takes the harmonic-mean throughput with a volatility discount and a safety factor, picks the highest
throughput-feasible tier, limits it with a graduated buffer cap, and steps down one tier under high volatility. It
uses the environment's own download model. RobustMPC follows Pensieve's `test/mpc.py`, with the environment's unit
factor so its forward model matches the environment it runs in.


In [ ]:
def expert_label(env, cfg):
    recent = np.asarray(env.tp_hist[-cfg.K:], dtype=np.float64)
    harm = cfg.K / np.sum(1.0 / np.maximum(recent, 0.1))
    cv = float(np.std(recent) / (np.mean(recent) + 1e-8))
    safe = harm / (1.0 + cv)
    committed = max(safe * cfg.EXPERT_SAFETY, 1e-6)
    if cfg.LADDER == "relative":
        sizes = [m * safe * env.CHUNK_DUR / 8.0 for m in cfg.TIER_MULT]
    elif cfg.LADDER == "content":
        sizes = list(env._obs_sizes) if env._obs_sizes is not None else list(cfg.BASE_CHUNK_MB)   # manifest sizes
    else:
        sizes = list(cfg.BASE_CHUNK_MB)
    dl_est = [s * env.unit / committed for s in sizes]
    ok = [b for b in range(cfg.N_BITRATES) if dl_est[b] <= env.CHUNK_DUR]
    tp_tier = max(ok) if ok else 0
    frac = float(np.clip((env.buffer - cfg.EXPERT_BUF_LO) / (cfg.EXPERT_BUF_HI - cfg.EXPERT_BUF_LO), 0, 1))
    buf_tier = int(round(frac * (cfg.N_BITRATES - 1)))
    label = min(tp_tier, buf_tier)
    if cv > cfg.EXPERT_CV_HIGH:
        label = max(0, label - 1)
    return int(label)

def expert_mix_label(env, cfg, rng):
    a = expert_label(env, cfg)
    u = rng.random()
    if u < cfg.LABEL_MIX[0]:
        return a
    if u < cfg.LABEL_MIX[0] + cfg.LABEL_MIX[1]:
        return int(rng.integers(max(0, a - 1), min(cfg.N_BITRATES - 1, a + 1) + 1))
    return max(0, a - 1)

class MPCPolicy:
    # RobustMPC as in Pensieve's test/mpc.py, plus the environment's unit factor.
    def __init__(self, cfg):
        self.cfg = cfg
        self.combos = {h: np.array(list(itertools.product(range(cfg.N_BITRATES), repeat=h)))
                       for h in range(1, cfg.MPC_LOOKAHEAD + 1)}
        self.reset()
    def reset(self):
        self.past_errors, self.past_bw = [], []
    def act(self, obs, env):
        cfg = self.cfg
        recent = [t for t in env.tp_hist[-5:] if t > 0]
        hbw = len(recent) / sum(1.0 / t for t in recent) if recent else 1.0
        err = abs(self.past_bw[-1] - env.tp_hist[-1]) / max(env.tp_hist[-1], 1e-8) if self.past_bw else 0.0
        self.past_errors.append(err)
        fbw = max(hbw / (1.0 + max(self.past_errors[-5:])), 1e-8)
        self.past_bw.append(hbw)
        horizon = min(cfg.MPC_LOOKAHEAD, env.n_chunks - env.t)
        if horizon <= 0:
            return 0
        combos = self.combos[horizon]
        if cfg.LADDER == "content":
            table = env.future_sizes(horizon)            # RobustMPC reads the exact future sizes from the manifest
        else:
            if cfg.LADDER == "relative":
                exp_sizes = np.asarray(cfg.TIER_MULT) * fbw * env.CHUNK_DUR / 8.0
            else:
                exp_sizes = np.asarray(cfg.BASE_CHUNK_MB, dtype=float)
            table = np.tile(exp_sizes, (horizon, 1))
            table[0] = np.asarray(obs["_next_raw"], dtype=float)
        dl = table[np.arange(horizon)[None, :], combos] * env.unit / fbw
        buf = np.full(len(combos), env.buffer)
        reb = np.zeros(len(combos))
        for p in range(horizon):
            short = buf < dl[:, p]
            reb += np.where(short, dl[:, p] - buf, 0.0)
            buf = np.minimum(np.where(short, 0.0, buf - dl[:, p]) + env.CHUNK_DUR, env.MAX_BUF)
        q = np.asarray(cfg.QUALITY)[combos]
        prev = np.concatenate([np.full((len(combos), 1), cfg.QUALITY[env.last_br]), q[:, :-1]], axis=1)
        score = q.sum(1) - cfg.ALPHA * reb - cfg.BETA * np.abs(q - prev).sum(1)
        return int(combos[int(np.argmax(score)), 0])
print("expert and RobustMPC defined")

## 3. BC data from expert rollouts, and normalisation statistics

Statistics are computed from the raw features of these rollouts (one mean/std per feature group).


In [ ]:
def build_bc_data(cfg):
    env = ABREnv(cfg, norm=None, split="train", seed=cfg.SEED + 123)
    rng = np.random.default_rng(cfg.SEED + 7)
    rows = []
    for _ in range(cfg.BC_EPISODES):
        obs, done = env.reset(), False
        while not done:
            a = expert_mix_label(env, cfg, rng)
            rows.append((obs["_raw"], obs["last_bitrate"], obs["buffer_cat"], a, obs["buffer_s"]))
            obs, _, done, _ = env.step(a)
    raws = [r[0] for r in rows]
    norm = dict(
        remain_mean=float(np.mean([r["remain"] for r in raws])), remain_std=float(np.std([r["remain"] for r in raws])),
        size_mean=float(np.mean([r["sizes"] for r in raws])),   size_std=float(np.std([r["sizes"] for r in raws])),
        dl_mean=float(np.mean([r["dl"] for r in raws])),        dl_std=float(np.std([r["dl"] for r in raws])),
        tp_mean=float(np.mean([r["tp"] for r in raws])),        tp_std=float(np.std([r["tp"] for r in raws])),
    )
    obs_list = []
    for raw, lb, bc, _, bs in rows:
        o = normalise_obs(raw, norm); o.update(last_bitrate=lb, buffer_cat=bc, buffer_s=bs)
        obs_list.append(o)
    labels = np.array([r[3] for r in rows])
    return obs_list, labels, norm

if MODE == "run":
    t0 = time.time()
    bc_obs, bc_labels, NORM = build_bc_data(C)
    label_share = np.bincount(bc_labels, minlength=N_BITRATES) / len(bc_labels)
    print(f"BC samples: {len(bc_labels):,} in {time.time()-t0:.0f}s | label share per tier:", np.round(label_share, 3))
    json.dump(NORM, open(RUN_DIR / "norm_stats.json", "w"), indent=2)

## 4. Model


In [ ]:
class EmbeddingEncoder(nn.Module):
    def __init__(self, n, e=16, out=32):
        super().__init__(); self.embed = nn.Embedding(n, e); self.fc1 = nn.Linear(e, 32); self.fc2 = nn.Linear(32, out)
    def forward(self, x):
        return F.relu(self.fc2(F.relu(self.fc1(self.embed(x)))))

class FCEncoder(nn.Module):
    def __init__(self, i, out=32):
        super().__init__(); self.fc1 = nn.Linear(i, 32); self.fc2 = nn.Linear(32, out)
    def forward(self, x):
        return F.relu(self.fc2(F.relu(self.fc1(x))))

class CNN1DEncoder(nn.Module):
    def __init__(self, seq_len=8, out=32):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, 3, padding=1); self.conv2 = nn.Conv1d(64, 128, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1); self.fc = nn.Linear(128, out)
    def forward(self, x):
        x = F.relu(self.conv2(F.relu(self.conv1(x.unsqueeze(1)))))
        return F.relu(self.fc(self.pool(x).squeeze(-1)))

class LSTMEncoder(nn.Module):
    def __init__(self, hidden=32, layers=1):
        super().__init__(); self.lstm = nn.LSTM(1, hidden, layers, batch_first=True)
    def forward(self, x):
        _, (h, _) = self.lstm(x.unsqueeze(-1)); return h[-1]

class APPStateEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__(); d = cfg.ENC_OUT
        self.enc_bitrate = EmbeddingEncoder(cfg.N_BITRATES, cfg.EMBED_DIM, d)
        self.enc_buffer = EmbeddingEncoder(cfg.N_BUF_CATS, cfg.EMBED_DIM, d)
        self.enc_remain = FCEncoder(1, d); self.enc_sizes = FCEncoder(cfg.N_BITRATES, d)
        self.enc_dl = CNN1DEncoder(cfg.K, d); self.enc_tp = LSTMEncoder(d, cfg.LSTM_LAYERS)
        self.fc_app1 = nn.Linear(6 * d, cfg.FC_APP_DIM); self.fc_app2 = nn.Linear(cfg.FC_APP_DIM, cfg.FC_APP_DIM)
        self.dropout = nn.Dropout(cfg.BC_DROPOUT)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu"); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0, 0.1)
            elif isinstance(m, nn.LSTM):
                for n_, p in m.named_parameters():
                    if "weight_ih" in n_: nn.init.xavier_uniform_(p)
                    elif "weight_hh" in n_: nn.init.orthogonal_(p)
                    elif "bias" in n_:
                        nn.init.zeros_(p); k = p.size(0); p.data[k // 4:k // 2].fill_(1.0)
    def forward(self, lb, bc, rem, ns, dl, tp, *_):
        cat = torch.cat([self.enc_bitrate(lb), self.enc_buffer(bc), self.enc_remain(rem),
                         self.enc_sizes(ns), self.enc_dl(dl), self.enc_tp(tp)], dim=-1)
        f = self.dropout(F.relu(self.fc_app1(cat)))
        return F.relu(self.fc_app2(f))

class ABRActorCritic(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.encoder = APPStateEncoder(cfg)
        self.actor = nn.Sequential(nn.Linear(cfg.FC_APP_DIM, 64), nn.ReLU(), nn.Linear(64, cfg.N_BITRATES))
        self.critic = nn.Sequential(nn.Linear(cfg.FC_APP_DIM, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, *x):
        f = self.encoder(*x)
        return self.actor(f), self.critic(f).squeeze(-1)

class PensieveNet(nn.Module):
    # Pensieve (Mao et al., SIGCOMM 2017): last bitrate, buffer and remaining chunks -> dense 128 each; throughput and
    # download-time histories and the next chunk sizes -> 1-D conv, 128 filters of size 4; merged -> dense 128 -> head
    def __init__(self, cfg, out, future=False):
        super().__init__()
        self.nb, self.future = cfg.N_BITRATES, future
        self.fc_br, self.fc_buf, self.fc_rem = nn.Linear(1, 128), nn.Linear(1, 128), nn.Linear(1, 128)
        self.cv_tp, self.cv_dl, self.cv_sz = nn.Conv1d(1, 128, 4), nn.Conv1d(1, 128, 4), nn.Conv1d(1, 128, 4)
        n_in = 3 * 128 + 2 * 128 * (cfg.K - 3) + 128 * (cfg.N_BITRATES - 3)
        if future:                                    # COREL: 4 future throughputs -> 1-D conv, 128 filters of size 4
            self.cv_fut = nn.Conv1d(1, 128, 4); n_in += 128
        self.fc = nn.Linear(n_in, 128)
        self.head = nn.Linear(128, out)
    def forward(self, lb, bc, rem, ns, dl, tp, buf, fut=None, *_):
        br = (lb.float() / (self.nb - 1)).unsqueeze(-1)
        h = [F.relu(self.fc_br(br)), F.relu(self.fc_buf(buf)), F.relu(self.fc_rem(rem)),
             F.relu(self.cv_tp(tp.unsqueeze(1))).flatten(1), F.relu(self.cv_dl(dl.unsqueeze(1))).flatten(1),
             F.relu(self.cv_sz(ns.unsqueeze(1))).flatten(1)]
        if self.future:
            h.append(F.relu(self.cv_fut(fut.unsqueeze(1))).flatten(1))
        return self.head(F.relu(self.fc(torch.cat(h, -1))))

class PensieveActorCritic(nn.Module):
    # separate actor and critic networks (as in Pensieve); COREL adds a separate stall (cost) critic
    def __init__(self, cfg, cost_critic=False, future=False):
        super().__init__()
        self.actor_net = PensieveNet(cfg, cfg.N_BITRATES, future)
        self.critic_net = PensieveNet(cfg, 1, future)
        self.cost_net = PensieveNet(cfg, 1, future) if cost_critic else None
    def forward(self, *x):
        return self.actor_net(*x[:8]), self.critic_net(*x[:8]).squeeze(-1)
    def cost_value(self, *x):
        return self.cost_net(*x[:8]).squeeze(-1)

def make_model(cfg):
    algo = getattr(cfg, "ALGO", "dara")
    if algo == "dara":
        return ABRActorCritic(cfg)
    if algo == "corel_official":
        return PensieveActorCritic(cfg, cost_critic=True, future=True)
    return PensieveActorCritic(cfg, cost_critic=(algo == "corel"))

if MODE == "run":
    model = make_model(C).to(DEVICE)
    print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 5. Behaviour cloning

In [ ]:
def run_bc(model, obs_list, labels, cfg):
    hist = []
    if cfg.BC_EPOCHS <= 0 or len(labels) == 0:
        return hist
    idx = np.arange(len(labels)); rng = np.random.default_rng(cfg.SEED)
    rng.shuffle(idx); cut = int(0.9 * len(idx)); tr, va = idx[:cut], idx[cut:]
    opt = Adam(model.parameters(), lr=cfg.BC_LR, weight_decay=cfg.BC_WD)
    sch = CosineAnnealingLR(opt, T_max=cfg.BC_EPOCHS, eta_min=1e-5)
    y = torch.as_tensor(labels, dtype=torch.long, device=DEVICE)
    for ep in range(cfg.BC_EPOCHS):
        model.train(); rng.shuffle(tr); correct = n = 0
        for s in range(0, len(tr), cfg.BC_BATCH):
            b = tr[s:s + cfg.BC_BATCH]
            logits, _ = model(*obs_to_tensors([obs_list[i] for i in b]))
            yb = y[torch.as_tensor(b, device=DEVICE)]
            loss = F.cross_entropy(logits, yb, label_smoothing=cfg.BC_LABEL_SMOOTHING)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            correct += (logits.argmax(1) == yb).sum().item(); n += len(b)
        sch.step(); model.eval()
        with torch.no_grad():
            va_acc = np.mean([(model(*obs_to_tensors([obs_list[i] for i in va[s:s + 2048]]))[0].argmax(1)
                               == y[torch.as_tensor(va[s:s + 2048], device=DEVICE)]).float().mean().item()
                              for s in range(0, len(va), 2048)])
        hist.append(dict(epoch=ep + 1, train_acc=correct / n, val_acc=float(va_acc)))
        print(f"BC epoch {ep+1}: train_acc={correct/n:.3f} val_acc={va_acc:.3f}")
    return hist

if MODE == "run":
    bc_hist = run_bc(model, bc_obs, bc_labels, C)
    pd.DataFrame(bc_hist).to_csv(RUN_DIR / "bc_history.csv", index=False)
    bc_snapshot = copy.deepcopy(model.state_dict())

## 6. PPO fine-tuning (GAE, `RETURN_NORM_MODE`, `VALUE_DENORM`)

Rollouts are collected in `eval()` mode and updates run in `train()` mode.


In [ ]:
class RunningMeanStd:
    def __init__(self, beta=0.99):
        self.beta, self.mean, self.var, self.init = beta, 0.0, 1.0, False
    def update(self, x):
        m, v = float(np.mean(x)), float(np.var(x))
        if not self.init:
            self.mean, self.var, self.init = m, v, True
        else:
            self.mean = self.beta * self.mean + (1 - self.beta) * m
            self.var = self.beta * self.var + (1 - self.beta) * v
    def normalize(self, x):   return (x - self.mean) / (np.sqrt(self.var) + 1e-8)
    def denormalize(self, x): return x * (np.sqrt(self.var) + 1e-8) + self.mean

def run_ppo(model, cfg, norm, hook=None):
    # hook(ep, model) -> True if the surrogate was changed (then the critic is warmed up)
    log = []
    if cfg.PPO_EPISODES <= 0:
        return log, None
    if cfg.VALUE_DENORM and cfg.RETURN_NORM_MODE != "running":
        raise ValueError("VALUE_DENORM=1 requires RETURN_NORM_MODE='running'")
    opt = Adam(model.parameters(), lr=cfg.PPO_LR)
    env = ABREnv(cfg, norm, "train", seed=cfg.SEED)          # env.cfg is cfg: surrogate changes apply immediately
    rms = RunningMeanStd() if cfg.RETURN_NORM_MODE == "running" else None
    v_raw = lambda v: float(rms.denormalize(v)) if (cfg.VALUE_DENORM and rms is not None and rms.init) else float(v)
    st = dict(obs=env.reset(), ep=0, lam=0.0, c2=float(cfg.PPO_C2))
    recent, recent_ref, t0, next_diag = [], [], time.time(), cfg.DIAG_EVERY
    corel = getattr(model, "cost_net", None) is not None
    rms_c = RunningMeanStd() if corel else None
    vc_raw = lambda v: float(rms_c.denormalize(v)) if (rms_c is not None and rms_c.init) else float(v)

    def collect():
        S, A, Rw, LP, V, D, RB, VC = [], [], [], [], [], [], [], []
        model.eval(); obs = st["obs"]
        for _ in range(cfg.PPO_ROLLOUT):
            with torch.no_grad():
                X1 = obs_to_tensors([obs])
                logits, val = model(*X1)
                if corel:
                    VC.append(vc_raw(model.cost_value(*X1).item()))
                dist = torch.distributions.Categorical(logits=logits)
                a = dist.sample()
            nxt, r, done, info = env.step(int(a.item()))
            if cfg.REWARD_MODE == "ref":
                r = info["ref"]                               # the intended per-chunk QoE u_t (no cap, clip or shaping)
            if cfg.LAGRANGE:
                r = r - st["lam"] * info["rebuffer"]
            S.append(obs); A.append(int(a.item())); Rw.append(r); RB.append(info["rebuffer"])
            LP.append(float(dist.log_prob(a).item())); V.append(v_raw(val.item())); D.append(float(done))
            if done:
                recent.append(env.ep_reward); recent_ref.append(env.ep_ref); st["ep"] += 1
                obs = env.reset()
            else:
                obs = nxt
        st["obs"] = obs
        with torch.no_grad():
            last_v = 0.0 if D[-1] else v_raw(model(*obs_to_tensors([obs]))[1].item())
        n = len(Rw); adv = np.zeros(n, np.float32); gae, nv = 0.0, last_v
        for t in reversed(range(n)):
            mask = 1.0 - D[t]
            delta = Rw[t] + cfg.PPO_GAMMA * nv * mask - V[t]
            gae = delta + cfg.PPO_GAMMA * cfg.PPO_LAMBDA * mask * gae
            adv[t] = gae; nv = V[t]
        values = np.array(V, np.float32); ret_raw = adv + values
        if cfg.RETURN_NORM_MODE == "batch":
            ret_t = (ret_raw - ret_raw.mean()) / (ret_raw.std() + 1e-8)
        elif cfg.RETURN_NORM_MODE == "raw":
            ret_t = ret_raw
        else:
            rms.update(ret_raw); ret_t = rms.normalize(ret_raw)
        ev = float(1 - np.var(ret_raw - values) / (np.var(ret_raw) + 1e-8))
        if cfg.LAGRANGE:
            st["lam"] = max(0.0, st["lam"] + cfg.LAG_LR * (float(np.mean(RB)) - cfg.LAG_TARGET))
        if corel:
            # stall critic: GAE on the per-chunk stall cost, its own running return normalisation
            with torch.no_grad():
                last_c = 0.0 if D[-1] else vc_raw(model.cost_value(*obs_to_tensors([obs])).item())
            advc = np.zeros(n, np.float32); g, nv = 0.0, last_c
            for t in reversed(range(n)):
                mask = 1.0 - D[t]
                delta = RB[t] + cfg.PPO_GAMMA * nv * mask - VC[t]
                g = delta + cfg.PPO_GAMMA * cfg.PPO_LAMBDA * mask * g
                advc[t] = g; nv = VC[t]
            retc = advc + np.array(VC, np.float32); rms_c.update(retc)
            st["advc"], st["retc"] = advc, rms_c.normalize(retc)
            # dual ascent on the measured stall per chunk
            st["lam"] = max(0.0, st["lam"] + cfg.COREL_LR * (float(np.mean(RB)) - cfg.COREL_TARGET))
        return S, A, LP, ret_t, adv, ev

    def update(S, A, LP, ret_t, adv, critic_only=False, critic_opt=None):
        n = len(A); adv_n = (adv - adv.mean()) / (adv.std() + 1e-8)
        if corel and not critic_only:
            # primal step: reward advantage minus lambda x stall advantage, rescaled (Safety-Gym PPO-Lagrangian form)
            ac = st["advc"]; ac_n = (ac - ac.mean()) / (ac.std() + 1e-8)
            adv_n = (adv_n - st["lam"] * ac_n) / (1.0 + st["lam"])
            rct = torch.as_tensor(st["retc"], dtype=torch.float32, device=DEVICE)
        X = obs_to_tensors(S)
        at = torch.as_tensor(A, device=DEVICE); olp = torch.as_tensor(LP, device=DEVICE)
        rt = torch.as_tensor(ret_t, dtype=torch.float32, device=DEVICE)
        advt = torch.as_tensor(adv_n, dtype=torch.float32, device=DEVICE)
        model.train(); ents = []
        for _ in range(cfg.PPO_EPOCHS):
            perm = torch.randperm(n, device=DEVICE)
            for s_ in range(0, n, cfg.PPO_BATCH):
                b = perm[s_:s_ + cfg.PPO_BATCH]
                logits, nv_ = model(*[x[b] for x in X])
                if critic_only:
                    loss = F.mse_loss(nv_, rt[b])
                    critic_opt.zero_grad(); loss.backward(); critic_opt.step()
                    continue
                dist = torch.distributions.Categorical(logits=logits)
                ratio = torch.exp(dist.log_prob(at[b]) - olp[b])
                pl = -torch.min(ratio * advt[b], torch.clamp(ratio, 1 - cfg.PPO_CLIP, 1 + cfg.PPO_CLIP) * advt[b]).mean()
                ent = dist.entropy().mean()
                loss = pl + cfg.PPO_C1 * F.mse_loss(nv_, rt[b]) - st["c2"] * ent
                if corel:
                    loss = loss + cfg.PPO_C1 * F.mse_loss(model.cost_value(*[x[b] for x in X]), rct[b])
                opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.PPO_MAX_GRAD); opt.step()
                ents.append(ent.item())
        return float(np.mean(ents)) if ents else np.nan

    while st["ep"] < cfg.PPO_EPISODES:
        S, A, LP, ret_t, adv, ev = collect()
        ent = update(S, A, LP, ret_t, adv)
        if cfg.ENT_ADAPT:
            st["c2"] = float(np.clip(np.exp(np.log(st["c2"]) + cfg.ENT_LR * (cfg.ENT_TARGET - ent)), 1e-4, 1.0))
        log.append(dict(episode=st["ep"], mean_reward_20=float(np.mean(recent[-20:])) if recent else np.nan,
                        mean_ref_20=float(np.mean(recent_ref[-20:])) if recent_ref else np.nan,
                        explained_variance=ev, entropy=ent, c2=st["c2"], lam=st["lam"],
                        rn_cap=float(cfg.RN_CAP), quality_k=float(cfg.QUALITY_K), rebuf_scale=float(cfg.REBUF_SCALE)))
        if len(log) % 50 == 0:
            print(f"ep {st['ep']:>6} | R20 {log[-1]['mean_reward_20']:8.2f} | EV {ev:6.3f} | H {ent:.3f} | "
                  f"c={cfg.RN_CAP:g} K={cfg.QUALITY_K:g} rho={cfg.REBUF_SCALE:g} c2={st['c2']:.3f} lam={st['lam']:.2f} | {time.time()-t0:.0f}s")
        if hook is not None and st["ep"] >= next_diag and st["ep"] < cfg.PPO_EPISODES:
            next_diag += cfg.DIAG_EVERY
            if hook(st["ep"], model):
                # critic warm-up on the new surrogate: actor and trunk frozen, value head only
                copt = Adam(model.critic.parameters(), lr=cfg.PPO_LR)
                for _ in range(cfg.WARMUP_UPDATES):
                    S, A, LP, ret_t, adv, ev = collect()
                    update(S, A, LP, ret_t, adv, critic_only=True, critic_opt=copt)
                log[-1]["warmup_ev"] = ev
    return log, rms

def run_a3c(model, cfg, norm):
    """Official-recipe training of COREL / Pensieve, re-implemented from the released TensorFlow code:
    16 agents each send up to A3C_SEQ steps per epoch; gradients of all agents are computed with the same parameters and
    applied one after another with RMSProp (decay 0.9, eps 1e-10); actor objective sum log pi(a|s) * (-td) + beta * sum p log p;
    critic and stall critic: mean squared error to n-step targets (bootstrapped from the last state's own estimate, as in
    the released code); the best checkpoint by validation reward is kept and returned."""
    log = []
    nA, constrained = cfg.N_BITRATES, cfg.ALGO == "corel_official"
    br = [b * 8.0 / ABREnv.CHUNK_DUR for b in cfg.BASE_CHUNK_MB]          # nominal tier bitrates, Mbps
    rms = lambda ps, lr: torch.optim.RMSprop(ps, lr=lr, alpha=0.9, eps=1e-10)
    nets = [(model.actor_net, rms(model.actor_net.parameters(), cfg.A3C_ACTOR_LR)),
            (model.critic_net, rms(model.critic_net.parameters(), cfg.A3C_CRITIC_LR))]
    if constrained:
        nets.append((model.cost_net, rms(model.cost_net.parameters(), cfg.A3C_CRITIC_LR)))
    params = [list(n.parameters()) for n, _ in nets]
    rng = np.random.default_rng(cfg.SEED + 11)
    envs = [ABREnv(cfg, norm, "train", seed=cfg.SEED + 7919 * i) for i in range(cfg.A3C_AGENTS)]
    obs = [e.reset() for e in envs]
    ep_r = [0.0] * len(envs)
    mu, mu_hist = float(cfg.COREL_MU0), [float(cfg.COREL_MU0)]
    best_score, best_epoch, best_stall, best_state = -np.inf, 0, np.nan, None
    recent, recent_ref, n_ep, epoch, t0 = [], [], 0, 0, time.time()
    G = cfg.PPO_GAMMA

    def validate(mu):
        env = ABREnv(cfg, norm, "eval", seed=cfg.EVAL_SEED + 500); o = env._state()
        scores, stalls = [], []
        for ep in range(cfg.A3C_VALID_EPISODES):
            if ep > 0: o = env.reset()
            done, rs = False, []
            while not done:
                a = int(policy_probs([o], model)[0].argmax()); prev = env.last_br
                o, _, done, info = env.step(a)
                rs.append((br[a] - mu * info["rebuffer"] - abs(br[a] - br[prev])) / cfg.A3C_STATE_SCALE)
            scores.append(np.mean(rs)); stalls.append(env.ep_rebuffer)
        return float(np.mean(scores)), float(np.mean(stalls))

    while n_ep < cfg.PPO_EPISODES:
        stage = min(int(len(cfg.A3C_ENTROPY) * n_ep / cfg.PPO_EPISODES), len(cfg.A3C_ENTROPY) - 1)
        beta = float(cfg.A3C_ENTROPY[stage])
        model.eval(); batches = []
        for i, e in enumerate(envs):
            S, A, R, U, done = [], [], [], [], False
            while len(R) < cfg.A3C_SEQ and not done:
                with torch.no_grad():
                    p = torch.softmax(model.actor_net(*obs_to_tensors([obs[i]])), -1)[0].cpu().numpy().astype(float)
                a = int(rng.choice(nA, p=p / p.sum())); prev = e.last_br
                nxt, _, done, info = e.step(a)
                r = (br[a] - mu * info["rebuffer"] - abs(br[a] - br[prev])) / cfg.A3C_STATE_SCALE
                S.append(obs[i]); A.append(a); R.append(r); U.append(info["rebuffer"]); ep_r[i] += r
                if done:
                    recent.append(ep_r[i]); recent_ref.append(e.ep_ref); ep_r[i] = 0.0; n_ep += 1
                    obs[i] = e.reset()
                else:
                    obs[i] = nxt
            batches.append((S, A, np.array(R), np.array(U), done))
        # gradients of every agent at the same parameters, then applied in turn (synchronous A3C as released)
        grads, evs, ents = [], [], []
        model.train()
        for S, A, R, U, term in batches:
            X = obs_to_tensors(S); n = len(A)
            probs = torch.softmax(model.actor_net(*X), -1)
            v = model.critic_net(*X).squeeze(-1)
            vn = v.detach().cpu().numpy()
            Rt = np.zeros(n); Rt[-1] = 0.0 if term else vn[-1]
            for t in reversed(range(n - 1)):
                Rt[t] = R[t] + G * Rt[t + 1]
            td = torch.as_tensor(Rt - vn, dtype=torch.float32, device=DEVICE)
            at = torch.as_tensor(A, device=DEVICE)
            obj = torch.sum(torch.log(probs.gather(1, at[:, None]).squeeze(1)) * (-td)) + \
                  beta * torch.sum(probs * torch.log(probs + 1e-6))
            losses = [obj, F.mse_loss(v, torch.as_tensor(Rt, dtype=torch.float32, device=DEVICE))]
            if constrained:
                u = model.cost_net(*X).squeeze(-1); un = u.detach().cpu().numpy()
                Ut = np.zeros(n); Ut[-1] = 0.0 if term else un[-1]
                for t in reversed(range(n - 1)):
                    Ut[t] = U[t] + Ut[t + 1]                  # undiscounted stall-time target, as released
                losses.append(F.mse_loss(u, torch.as_tensor(Ut, dtype=torch.float32, device=DEVICE)))
            grads.append([[None if g is None else g.detach().clone()
                           for g in torch.autograd.grad(L, ps, allow_unused=True, retain_graph=True)]
                          for L, ps in zip(losses, params)])
            evs.append(1.0 - np.var(Rt - vn) / (np.var(Rt) + 1e-8))
            ents.append(float(-(probs * torch.log(probs + 1e-6)).sum(1).mean().item()))
        for gset in grads:
            for (net, opt), ps, gs in zip(nets, params, gset):
                for p_, g_ in zip(ps, gs):
                    p_.grad = g_
                opt.step(); opt.zero_grad(set_to_none=True)
        epoch += 1
        row = dict(episode=n_ep, epoch=epoch, mean_reward_20=float(np.mean(recent[-20:])) if recent else np.nan,
                   mean_ref_20=float(np.mean(recent_ref[-20:])) if recent_ref else np.nan,
                   explained_variance=float(np.mean(evs)), entropy=float(np.mean(ents)), c2=beta, lam=mu, mu=mu,
                   rn_cap=float(cfg.RN_CAP), quality_k=float(cfg.QUALITY_K), rebuf_scale=float(cfg.REBUF_SCALE))
        if (epoch - 1) % cfg.A3C_VALID_EVERY == 0:
            score, stall = validate(mu)
            row.update(valid_score=score, valid_stall=stall)
            if score > best_score:
                best_score, best_epoch, best_stall = score, epoch, stall
                best_state = copy.deepcopy(model.state_dict())
            if constrained and epoch > 1 and (epoch - 1) % cfg.COREL_MU_EVERY == 0:
                mu = max(mu + cfg.COREL_MU_LR * (best_stall - cfg.COREL_EPS), 0.0)
                mu_hist.append(mu); best_score = -np.inf
                row["mu_updated"] = mu
            print(f"epoch {epoch:>5} | ep {n_ep:>6} | ref20 {row['mean_ref_20']:8.2f} | valid {score:7.3f} stall {stall:6.2f}s | "
                  f"mu {mu:7.2f} | beta {beta:g} | {time.time()-t0:.0f}s")
        log.append(row)
        if constrained and len(mu_hist) > 1 and epoch - best_epoch > cfg.A3C_PATIENCE and abs(mu_hist[-1] - mu_hist[-2]) < 0.01:
            print("early stop (mu converged)"); break
    if best_state is not None:
        model.load_state_dict(best_state)                     # the best checkpoint, as in the released code
    model.eval()
    return log, None


## 7. Diagnostic state set (shared across runs on the same environment)

States are collected with a fixed behaviour policy (the expert with 30% random actions), **not** the policy being diagnosed, so every checkpoint on the same environment is judged on identical states. They are stratified over throughput level × volatility × buffer × trend and cached under `OUT_ROOT/diag_states/`. The set depends only on environment settings (ladder, units, size consistency, buffer cap), not on the reward.

In [ ]:
def snap_features(env, cfg):
    recent = np.asarray(env.tp_hist[-cfg.K:], dtype=np.float64)
    harm = cfg.K / np.sum(1.0 / np.maximum(recent, 0.1))
    cv = float(np.std(recent) / (np.mean(recent) + 1e-8))
    tr = float(np.log(max(env.tp_hist[-1], 0.1)) - np.log(max(np.mean(env.tp_hist[-4:-1]), 0.1)))
    trend = "degrading" if tr < -0.2 else ("recovering" if tr > 0.2 else "steady")
    return dict(harm=float(harm), cv=cv, buffer=float(env.buffer), trend=trend, trend_val=tr,
                scenario=env.scenario, t=env.t, n_chunks=env.n_chunks)

def collect_diag_states(cfg):
    if cfg.LADDER == "relative":
        lad = cfg.TIER_MULT
    elif cfg.LADDER == "content":
        lad = list(cfg.BASE_CHUNK_MB) + [cfg.CONTENT_CAP, cfg.CONTENT_SLOW, cfg.CONTENT_VAR_SCALE,
                                         int(load_content_model(cfg)["md5"], 16) % 1_000_000]
    else:
        lad = cfg.BASE_CHUNK_MB
    lad_h = hashlib.md5(json.dumps([round(float(x), 6) for x in lad]).encode()).hexdigest()[:8]
    key = (f"{cfg.LADDER}_u{cfg.UNITS_FIX}_sc{cfg.SIZE_CONSISTENT}_{cfg.DL_MODEL}_buf{cfg.MAX_BUF}_L{lad_h}"
           f"_ds{cfg.DIAG_SEED}_n{cfg.N_DIAG}_h{max(cfg.H_LIST)}")
    # network part of the key (empty for the default generator mix, so earlier caches stay valid)
    if cfg.NET_SOURCE == "trace":
        key += "_tr" + hashlib.md5(f"{cfg.TRACE_FILE}|{cfg.TRACE_PREFIX}|{float(cfg.NET_SCALE)}".encode()).hexdigest()[:8]
        if float(getattr(cfg, "TRACE_HOLDOUT_FRAC", 0.0) or 0.0) > 0:
            key += f"_ho{float(cfg.TRACE_HOLDOUT_FRAC):g}s{int(cfg.TRACE_HOLDOUT_SEED)}"
    elif cfg.NET_SCENARIOS or float(cfg.NET_SCALE) != 1.0:
        key += "_net" + hashlib.md5(json.dumps([sorted(cfg.NET_SCENARIOS), float(cfg.NET_SCALE)]).encode()).hexdigest()[:8]
    cache = Path(cfg.OUT_ROOT) / "diag_states" / f"{key}.pkl"
    if cache.exists():
        print("loading cached diagnostic states:", cache)
        return pickle.load(open(cache, "rb")), key
    env = ABREnv(cfg, norm=None, split="eval", seed=cfg.DIAG_SEED)
    brng = np.random.default_rng(cfg.DIAG_SEED + 5)
    hmax, pool = max(cfg.H_LIST), []
    while len(pool) < cfg.N_DIAG * cfg.POOL_FACTOR:
        done = False
        while not done:
            if env.n_chunks - env.t > hmax and brng.random() < cfg.SNAP_PROB:
                pool.append((copy.deepcopy(env), snap_features(env, cfg)))
            a = int(brng.integers(cfg.N_BITRATES)) if brng.random() < cfg.BEHAV_RANDOM else expert_label(env, cfg)
            _, _, done, _ = env.step(a)
        env.reset()
    lh = np.log([f["harm"] for _, f in pool])
    q1, q2 = np.quantile(lh, [1 / 3, 2 / 3]); cv_med = np.median([f["cv"] for _, f in pool])
    for (_, f), l in zip(pool, lh):
        f["level"] = "low" if l < q1 else ("mid" if l < q2 else "high")
        f["vol"] = "volatile" if f["cv"] >= cv_med else "stable"
        f["buf"] = "low_buf" if f["buffer"] < 5.0 else "high_buf"
        f["cell"] = f"{f['level']}|{f['vol']}|{f['buf']}|{f['trend']}"
    cells = {}
    for i, (_, f) in enumerate(pool):
        cells.setdefault(f["cell"], []).append(i)
    quota = int(np.ceil(cfg.N_DIAG / len(cells)))
    rng = np.random.default_rng(cfg.DIAG_SEED + 9)
    chosen = []
    for c in sorted(cells):
        idx = cells[c]; chosen += list(rng.choice(idx, size=min(quota, len(idx)), replace=False))
    rest = [i for i in range(len(pool)) if i not in set(chosen)]
    if len(chosen) < cfg.N_DIAG:
        chosen += list(rng.choice(rest, size=min(cfg.N_DIAG - len(chosen), len(rest)), replace=False))
    chosen = sorted(chosen[:cfg.N_DIAG]) if len(chosen) > cfg.N_DIAG else sorted(chosen)
    states = [pool[i] for i in chosen]
    cache.parent.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_suffix(f".tmp{os.getpid()}")
    pickle.dump(states, open(tmp, "wb")); os.replace(tmp, cache)
    return states, key

if MODE == "run":
    DIAG, DIAG_KEY = collect_diag_states(C)
    diag_meta = pd.DataFrame([f for _, f in DIAG])
    print(f"{len(DIAG)} diagnostic states | {diag_meta['cell'].nunique()} cells | key={DIAG_KEY}")
    print(diag_meta.groupby(["level", "buf"]).size().unstack(fill_value=0))

## 8. Counterfactual rollouts (common random numbers)

For every state *sᵢ*, action *a* and repetition *r*, the environment is deep-copied and its RNG reseeded with the same seed for all five actions, so every action faces the identical future throughput trace and chunk-size draws.

* **hold-a** — take *a* for *H* steps → fixed-tier value `Qh`, reference return `Rh`, raw rebuffering, cap/clip saturation.
* **follow-π** — take *a*, then the greedy policy → `Qf` (≈ Q^π, used for policy–value agreement *A*).

All environments for one state (5 actions × `N_REP`) step in lockstep so the policy is called in batches.

In [ ]:
def policy_probs(obs_list, model):
    model.eval()
    with torch.no_grad():
        logits, _ = model(*obs_to_tensors(obs_list))
        return torch.softmax(logits, -1).cpu().numpy()

def rebind(e, cfg, norm):
    # cached snapshots carry the cfg of the run that created them: always rebind to THIS run's reward/env settings
    e.cfg, e.norm, e.unit, e.MAX_BUF = cfg, norm, (8.0 if cfg.UNITS_FIX else 1.0), cfg.MAX_BUF
    if cfg.LADDER == "content":
        e.content = load_content_model(cfg)
    return e

def counterfactual(states, cfg, norm, model, H, modes=("hold", "follow")):
    n, A_, R_ = len(states), cfg.N_BITRATES, cfg.N_REP
    out = {k: np.zeros((n, A_)) for k in ["Qh", "Rq", "Rrb", "Rsw", "rebh", "caph", "cliph", "Qf", "rebf"]}
    probs = np.zeros((n, A_))
    for i, (snap, _) in enumerate(states):
        base = cfg.DIAG_SEED * 1_000_003 + i * 97
        e0 = rebind(copy.deepcopy(snap), cfg, norm); e0._rng = np.random.default_rng(base + 99)
        if model is not None:
            probs[i] = policy_probs([e0._state()], model)[0]
        for mode in modes:
            envs, first = [], []
            for a in range(A_):
                for r in range(R_):
                    e = rebind(copy.deepcopy(snap), cfg, norm); e._rng = np.random.default_rng(base + r)
                    envs.append(e); first.append(a)
            G = np.zeros(len(envs)); Gq = np.zeros(len(envs)); Grb = np.zeros(len(envs)); Gsw = np.zeros(len(envs))
            reb = np.zeros(len(envs))
            cap = np.zeros(len(envs)); clp = np.zeros(len(envs))
            alive = np.ones(len(envs), bool); acts = np.array(first); disc = 1.0
            for h in range(H):
                nxt = [None] * len(envs)
                for j, e in enumerate(envs):
                    if not alive[j]: continue
                    o, rw, d, info = e.step(int(acts[j]))
                    G[j] += disc * rw; reb[j] += info["rebuffer"]
                    Gq[j] += disc * info["ref_q"]; Grb[j] += disc * info["rebuffer"]; Gsw[j] += disc * info["ref_sw"]
                    cap[j] += info["capped"]; clp[j] += info["clipped"]
                    if d: alive[j] = False
                    else: nxt[j] = o
                disc *= cfg.DIAG_GAMMA
                if mode == "follow" and alive.any() and h < H - 1:
                    live = np.where(alive)[0]
                    acts[live] = policy_probs([nxt[j] for j in live], model).argmax(1)
            sh = (A_, R_)
            if mode == "hold":
                out["Qh"][i] = G.reshape(sh).mean(1)
                out["Rq"][i] = Gq.reshape(sh).mean(1); out["Rrb"][i] = Grb.reshape(sh).mean(1); out["Rsw"][i] = Gsw.reshape(sh).mean(1)
                out["rebh"][i] = reb.reshape(sh).mean(1); out["caph"][i] = cap.reshape(sh).mean(1)
                out["cliph"][i] = clp.reshape(sh).mean(1)
            else:
                out["Qf"][i] = G.reshape(sh).mean(1)
                out["rebf"][i] = reb.reshape(sh).mean(1)
        if (i + 1) % 50 == 0:
            print(f"  H={H}: {i+1}/{n} states")
    out["probs"] = probs
    return out

## 9. Signals and triage

| Signal | Definition |
|---|---|
| **A** | fraction of states where the policy's greedy action is within `AGREE_TOL` × (that state's Qf range) of max_a Qf; exact-argmax version also reported |
| **D** | max_a fraction of states whose hold-best action a*ᵢ = argmax Qh equals *a* |
| **ΔQ** | mean over states of (best − second-best Qh) / (best − worst Qh): scale-free, in [0, 1] |
| **M** | fraction of states where a*ᵢ loses to the best action under the **reference** objective by > `REF_MARGIN` (also reported for every weight in `REF_ALPHA_GRID`) |
| M_rebuf | fraction where a*ᵢ rebuffers > `REBUF_MARGIN` s more than the least-rebuffering action |
| cap_sat | fraction where a*ᵢ hits the RN_CAP bound at least once |
| SCAS_π, SCAS_Q | mean pairwise Jensen–Shannon divergence (base 2) between condition cells, for π and for argmax Qh |
| H(π) | mean policy entropy on the diagnostic states |

Triage rule: `A < A_MIN → OPT_FAILURE`; else `D ≥ D_HIGH → (M ≥ M_HIGH ? REWARD_INDUCED : ENV_DEGENERATE)`; else `HEALTHY`.

In [ ]:
def js2(p, q):
    p = np.asarray(p, float) + 1e-12; q = np.asarray(q, float) + 1e-12
    p /= p.sum(); q /= q.sum(); m = 0.5 * (p + q)
    return float(0.5 * np.sum(p * np.log2(p / m)) + 0.5 * np.sum(q * np.log2(q / m)))

def ref_returns(cf, cfg, ref_alpha=None):
    a = cfg.REF_ALPHA if ref_alpha is None else ref_alpha
    return cf["Rq"] - a * cf["Rrb"] - cfg.REF_BETA * cf["Rsw"]

def core_signals(cf, idx, cfg, ref_alpha=None):
    Qh, Qf, rebh, caph, P = (cf[k][idx] for k in ["Qh", "Qf", "rebh", "caph", "probs"])
    Rh = ref_returns(cf, cfg, ref_alpha)[idx]
    a_star = Qh.argmax(1); a_pi = P.argmax(1); n = len(idx)
    rng_f = Qf.max(1) - Qf.min(1); rng_h = Qh.max(1) - Qh.min(1)
    A_strict = float(np.mean(a_pi == Qf.argmax(1)))
    A = float(np.mean(Qf[np.arange(n), a_pi] >= Qf.max(1) - cfg.AGREE_TOL * rng_f - 1e-9))
    counts = np.bincount(a_star, minlength=cfg.N_BITRATES); D = float(counts.max() / n)
    srt = np.sort(Qh, 1); dQ = float(np.mean((srt[:, -1] - srt[:, -2]) / (rng_h + 1e-8)))
    M = float(np.mean(Rh[np.arange(n), a_star] < Rh.max(1) - cfg.REF_MARGIN))
    M_reb = float(np.mean(rebh[np.arange(n), a_star] > rebh.min(1) + cfg.REBUF_MARGIN))
    cap_sat = float(np.mean(caph[np.arange(n), a_star] > 0))
    ent = float(np.mean(-(P * np.log(P + 1e-12)).sum(1)))
    return dict(A=A, A_strict=A_strict, D=D, dominant_action=int(counts.argmax()), dQ=dQ, M=M,
                M_rebuf=M_reb, cap_sat=cap_sat, entropy=ent,
                pi_mode_share=float(np.bincount(a_pi, minlength=cfg.N_BITRATES).max() / n))

def scas(cf, meta, cfg):
    cols = [c.strip() for c in cfg.SCAS_GROUP.split(",")]
    cells = meta[cols].astype(str).agg("|".join, axis=1).values; ok = [c for c in np.unique(cells) if np.sum(cells == c) >= cfg.SCAS_MIN_STATES]
    a_star = cf["Qh"].argmax(1)
    Pc = [cf["probs"][cells == c].mean(0) for c in ok]
    Qc = [np.bincount(a_star[cells == c], minlength=cfg.N_BITRATES) + 0.5 for c in ok]
    pairs = list(itertools.combinations(range(len(ok)), 2))
    if not pairs:
        return np.nan, np.nan, len(ok)
    return (max(0.0, float(np.mean([js2(Pc[i], Pc[j]) for i, j in pairs]))),
            max(0.0, float(np.mean([js2(Qc[i], Qc[j]) for i, j in pairs]))), len(ok))

def triage(s, cfg):
    if s["A"] < cfg.A_MIN: return "OPT_FAILURE"
    if s["D"] >= cfg.D_HIGH: return "REWARD_INDUCED" if s["M"] >= cfg.M_HIGH else "ENV_DEGENERATE"
    return "HEALTHY"

def full_signals(cf, meta, cfg):
    n = len(meta); idx = np.arange(n); s = core_signals(cf, idx, cfg)
    s["SCAS_pi"], s["SCAS_Q"], s["scas_cells"] = scas(cf, meta, cfg)
    rng = np.random.default_rng(cfg.SEED + 11); boots = {k: [] for k in ["A", "D", "dQ", "M"]}; labels = []
    for _ in range(cfg.N_BOOT):
        b = rng.integers(0, n, n); bs = core_signals(cf, b, cfg)
        for k in boots: boots[k].append(bs[k])
        labels.append(triage(bs, cfg))
    for k, v in boots.items():
        s[f"{k}_lo"], s[f"{k}_hi"] = (float(x) for x in np.quantile(v, [0.025, 0.975]))
    for ra in cfg.REF_ALPHA_GRID:
        s[f"M_refalpha_{ra}"] = core_signals(cf, idx, cfg, ref_alpha=ra)["M"]
    s["triage"] = triage(s, cfg)
    s["triage_stability"] = float(np.mean(np.array(labels) == s["triage"]))
    return s

## 10. Independent ground truth (episode level)

The **reference objective** is the standard linear QoE with the same quality scale and the same per-second rebuffering weight (α / chunk duration), but **uncapped, unclipped and without progress weighting**. It is fixed before any run and never changes with the regime, so it plays the role of the "intended" objective against which the training reward is judged.

Computed without the diagnostic signals: full test episodes (seed `EVAL_SEED`, identical across policies; on a trace network with `TRACE_HOLDOUT_FRAC > 0` they use only the held-out traces) for the greedy policy, each fixed tier held for the whole episode, and RobustMPC.

* `OPT_FAILURE` if the policy's shaped reward is more than `GT_FAIL_MARGIN` below the best fixed tier;
* else, if one action accounts for ≥ `GT_DOMINANT` of the policy's decisions: `ENV_DEGENERATE` when the reward-best and reference-best fixed tiers agree, `REWARD_INDUCED` when they differ;
* else `HEALTHY`.

The construction check prints whether the regime was built as intended (e.g. `reward_induced_*` needs reward-best tier 4 and reference-best tier < 4).

In [ ]:
def run_episodes(cfg, norm, n, seed, kind, model=None, tier=None, split="eval"):
    env = ABREnv(cfg, norm, split, seed=seed); mpc = MPCPolicy(cfg) if kind == "mpc" else None
    rows, acts = [], []
    obs = env._state()                      # first episode is already reset by __init__
    for ep in range(n):
        if ep > 0:
            obs = env.reset()
        done = False
        if mpc: mpc.reset()
        while not done:
            if kind == "policy": a = int(policy_probs([obs], model)[0].argmax())
            elif kind == "tier": a = tier
            else: a = mpc.act(obs, env)
            acts.append(a); obs, _, done, _ = env.step(a)
        rows.append(dict(reward=env.ep_reward, ref=env.ep_ref, rebuffer=env.ep_rebuffer, chunks=env.n_chunks))
    return pd.DataFrame(rows), np.array(acts)

def ground_truth(cfg, norm, model):
    res = {}
    pol, pol_acts = run_episodes(cfg, norm, cfg.GT_EPISODES, cfg.EVAL_SEED, "policy", model=model, split="test")
    res["policy"] = pol
    for b in range(cfg.N_BITRATES):
        res[f"tier{b}"], _ = run_episodes(cfg, norm, cfg.GT_EPISODES, cfg.EVAL_SEED, "tier", tier=b, split="test")
    res["robustmpc"], mpc_acts = run_episodes(cfg, norm, cfg.GT_EPISODES, cfg.EVAL_SEED, "mpc", split="test")
    table = pd.DataFrame({k: dict(reward=v.reward.mean(), ref=v.ref.mean(), rebuffer=v.rebuffer.mean())
                          for k, v in res.items()}).T
    tiers = [f"tier{b}" for b in range(cfg.N_BITRATES)]
    best_r = int(np.argmax(table.loc[tiers, "reward"].values)); best_ref = int(np.argmax(table.loc[tiers, "ref"].values))
    best_fixed = table.loc[f"tier{best_r}", "reward"]
    share = np.bincount(pol_acts, minlength=cfg.N_BITRATES) / len(pol_acts)
    p_vs_best = np.nan
    if sps is not None:
        d = res["policy"].reward.values - res[f"tier{best_r}"].reward.values
        p_vs_best = float(sps.ttest_rel(res["policy"].reward.values, res[f"tier{best_r}"].reward.values).pvalue) if np.std(d) > 0 else 1.0
    if table.loc["policy", "reward"] < best_fixed - cfg.GT_FAIL_MARGIN * abs(best_fixed):
        gt = "OPT_FAILURE"
    elif share.max() >= cfg.GT_DOMINANT:
        gt = "ENV_DEGENERATE" if best_r == best_ref else "REWARD_INDUCED"
    else:
        gt = "HEALTHY"
    info = dict(gt_label=gt, reward_best_tier=best_r, ref_best_tier=best_ref, policy_reward=table.loc["policy", "reward"],
                best_fixed_reward=best_fixed, policy_ref=table.loc["policy", "ref"], mpc_reward=table.loc["robustmpc", "reward"],
                policy_rebuffer=table.loc["policy", "rebuffer"], policy_mode_share=float(share.max()),
                policy_mode_tier=int(share.argmax()), p_policy_vs_best_fixed=p_vs_best,
                policy_tier_share=json.dumps(np.round(share, 4).tolist()),
                mpc_tier_share=json.dumps(np.round(np.bincount(mpc_acts, minlength=cfg.N_BITRATES) / len(mpc_acts), 4).tolist()))
    return table, info, res

## 10b. Landscape sweep (`MODE='landscape'`, no training)

D, M, ΔQ, SCAS_Q and cap saturation depend only on the reward and the environment, so they can be mapped over a grid of reward settings without training a policy. For each ladder scale (fixed ladder) the diagnostic states are collected once; for each reward setting the hold-a rollouts are re-run and full-episode fixed-tier simulations give the reward-best and reference-best constant tiers. Output: `OUT_ROOT/landscape/landscape_<ladder>_u<units>.csv` and two figures. Nothing here is used to choose detector thresholds.

In [ ]:
def landscape_signals(cf, meta, cfg, n_boot=200, seed=0):
    Qh, rebh, caph = cf["Qh"], cf["rebh"], cf["caph"]
    Rh = ref_returns(cf, cfg); n = len(Qh); A_ = cfg.N_BITRATES

    def sig(idx):
        q, r, rb, cp = Qh[idx], Rh[idx], rebh[idx], caph[idx]; m = len(idx)
        a_star = q.argmax(1); counts = np.bincount(a_star, minlength=A_)
        rng_h = q.max(1) - q.min(1); srt = np.sort(q, 1)
        return dict(D=float(counts.max() / m), dominant_action=int(counts.argmax()),
                    dQ=float(np.mean((srt[:, -1] - srt[:, -2]) / (rng_h + 1e-8))),
                    M=float(np.mean(r[np.arange(m), a_star] < r.max(1) - cfg.REF_MARGIN)),
                    M_rebuf=float(np.mean(rb[np.arange(m), a_star] > rb.min(1) + cfg.REBUF_MARGIN)),
                    cap_sat=float(np.mean(cp[np.arange(m), a_star] > 0)),
                    # intended-return regret of the reward-preferred action (continuous companion of M)
                    regret=float(np.mean(r.max(1) - r[np.arange(m), a_star])),
                    # freeze of the reward-preferred action minus that of the intended-best action (s over H chunks):
                    # > 0 means the reward prices freezes too low, < 0 too high
                    freeze_excess=float(np.mean(rb[np.arange(m), a_star] - rb[np.arange(m), r.argmax(1)])),
                    # DARA-L budget: freeze per chunk of the intended objective's best hold action, averaged over states
                    intended_freeze_per_chunk=float(np.mean(rb[np.arange(m), r.argmax(1)]) / cfg.H_MAIN))
    s = sig(np.arange(n))
    for ra in cfg.REF_ALPHA_GRID:
        s[f"M_refalpha_{ra}"] = float(np.mean(ref_returns(cf, cfg, ra)[np.arange(n), Qh.argmax(1)]
                                              < ref_returns(cf, cfg, ra).max(1) - cfg.REF_MARGIN))
    cols = [c.strip() for c in cfg.SCAS_GROUP.split(",")]
    cells = meta[cols].astype(str).agg("|".join, axis=1).values
    ok = [c for c in np.unique(cells) if np.sum(cells == c) >= cfg.SCAS_MIN_STATES]
    a_star = Qh.argmax(1)
    Qc = [np.bincount(a_star[cells == c], minlength=A_) + 0.5 for c in ok]
    pairs = list(itertools.combinations(range(len(ok)), 2))
    s["SCAS_Q"] = max(0.0, float(np.mean([js2(Qc[i], Qc[j]) for i, j in pairs]))) if pairs else np.nan
    rng = np.random.default_rng(seed); bD, bM = [], []
    for _ in range(n_boot):
        b = rng.integers(0, n, n); bs = sig(b); bD.append(bs["D"]); bM.append(bs["M"])
    s["D_lo"], s["D_hi"] = (float(x) for x in np.quantile(bD, [0.025, 0.975]))
    s["M_lo"], s["M_hi"] = (float(x) for x in np.quantile(bM, [0.025, 0.975]))
    return s

def fixed_tier_table(cfg):
    out = {}
    for b in range(cfg.N_BITRATES):
        df, _ = run_episodes(cfg, None, cfg.GT_EPISODES, cfg.EVAL_SEED, "tier", tier=b)
        out[b] = (df.reward.mean(), df.ref.mean(), df.rebuffer.mean())
    return out

if MODE == "landscape":
    LS_DIR = Path(OUT_ROOT) / "landscape"; LS_DIR.mkdir(parents=True, exist_ok=True)
    scales = LS_LADDER_SCALE if LADDER in ("fixed", "content") else [1.0]
    base_mb = [float(x) for x in BASE_CHUNK_MB]
    rows, t_all = [], time.time()
    for sc in scales:
        C.BASE_CHUNK_MB = [x * sc for x in base_mb]
        C.RN_CAP, C.QUALITY_K, C.QUALITY = RN_CAP, 0.0, list(QUALITY_BASE)
        states, key = collect_diag_states(C)
        meta = pd.DataFrame([f for _, f in states])
        configs = [(rc, kq, None, 1.0) for rc in LS_RN_CAP for kq in LS_QUALITY_K]
        configs += [(1e6, 0.0, qt, 1.0) for qt in LS_QTOP if abs(qt - QUALITY_BASE[-1]) > 1e-9]
        configs += [(rc, 0.0, None, float(rho)) for rc in LS_RN_CAP for rho in LS_REBUF_SCALE if abs(rho - 1.0) > 1e-9]
        for rc, kq, qt, rho in configs:
            t0 = time.time()
            c = copy.copy(C); c.RN_CAP, c.QUALITY_K, c.REBUF_SCALE = float(rc), float(kq), float(rho)
            q = list(4.0 * (1.0 - np.exp(-kq * np.array(TIER_MULT)))) if kq > 0 else list(QUALITY_BASE)
            if qt is not None:
                q[-1] = float(qt)
            c.QUALITY = q
            cf = counterfactual(states, c, None, None, H_MAIN, modes=("hold",))
            sgn = landscape_signals(cf, meta, c, n_boot=LS_BOOT, seed=SEED)
            ft = fixed_tier_table(c)
            rbest = int(np.argmax([ft[b][0] for b in ft])); fbest = int(np.argmax([ft[b][1] for b in ft]))
            row = dict(ladder=LADDER, units_fix=UNITS_FIX, dl_model=DL_MODEL, ladder_scale=sc, diag_key=key, rn_cap=rc, quality_k=kq, rebuf_scale=rho,
                       q_top=q[-1], quality=json.dumps([round(x, 3) for x in q]),
                       reward_best_tier=rbest, ref_best_tier=fbest, best_tiers_agree=int(rbest == fbest))
            row.update(sgn)
            for b in ft:
                row[f"tier{b}_reward"], row[f"tier{b}_ref"], row[f"tier{b}_rebuffer"] = ft[b]
            rows.append(row)
            print(f"scale={sc:<5} RN_CAP={rc:<8g} K_Q={kq:<5g} rho={rho:<4g} q_top={q[-1]:<5.2f} | D={sgn['D']:.3f} M={sgn['M']:.3f} "
                  f"regret={sgn['regret']:.3f} fx={sgn['freeze_excess']:+.2f} "
                  f"dQ={sgn['dQ']:.3f} cap={sgn['cap_sat']:.2f} | best tier reward/ref={rbest}/{fbest} | {time.time()-t0:.0f}s")
    LS = pd.DataFrame(rows)
    out_csv = LS_DIR / f"landscape_{LADDER}_u{UNITS_FIX}_{DL_MODEL}.csv"
    LS.to_csv(out_csv, index=False)
    for sig_name, ylab in (("D", "Action dominance D"), ("M", "Outcome misalignment M")):
        fig, axes = plt.subplots(1, len(scales), figsize=(3.6 * len(scales), 3.2), squeeze=False, sharey=True)
        for ax, sc in zip(axes[0], scales):
            sub = LS[(LS.rebuf_scale == 1.0) & ((LS.ladder_scale == sc) & (LS.q_top == QUALITY_BASE[-1]) | (LS.ladder_scale == sc) & (LS.quality_k > 0))]
            for rc, grp in sub.groupby("rn_cap"):
                grp = grp.sort_values("quality_k")
                ax.plot(grp.quality_k, grp[sig_name], marker="o", ms=3, label=f"RN_CAP={rc:g}")
            ax.set_xlabel("K_Q" + (f"  (ladder x{sc:g})" if LADDER in ("fixed", "content") else ""))
        axes[0][0].set_ylabel(ylab); axes[0][-1].legend(fontsize=6, frameon=False)
        fig.tight_layout(); fig.savefig(LS_DIR / f"landscape_{LADDER}_u{UNITS_FIX}_{DL_MODEL}_{sig_name}.png", dpi=200); plt.close(fig)
    if (LS.rebuf_scale != 1.0).any():
        for sig_name, ylab in (("M", "Outcome misalignment M"), ("regret", "Intended-return regret"), ("freeze_excess", "Freeze excess (s)")):
            fig, axes = plt.subplots(1, len(scales), figsize=(3.6 * len(scales), 3.2), squeeze=False, sharey=True)
            for ax, sc in zip(axes[0], scales):
                sub = LS[(LS.ladder_scale == sc) & (LS.quality_k == 0) & (LS.q_top == QUALITY_BASE[-1])]
                for rc, grp in sub.groupby("rn_cap"):
                    grp = grp.sort_values("rebuf_scale")
                    ax.plot(grp.rebuf_scale, grp[sig_name], marker="o", ms=3, label=f"RN_CAP={rc:g}")
                ax.set_xscale("log", base=2)
                ax.set_xlabel("Freeze price rho" + (f"  (ladder x{sc:g})" if LADDER in ("fixed", "content") else ""))
            axes[0][0].set_ylabel(ylab); axes[0][-1].legend(fontsize=6, frameon=False)
            fig.tight_layout(); fig.savefig(LS_DIR / f"landscape_{LADDER}_u{UNITS_FIX}_{DL_MODEL}_price_{sig_name}.png", dpi=200); plt.close(fig)
    print(f"landscape: {len(LS)} configs in {time.time()-t_all:.0f}s -> {out_csv}")

## 11. Train: PPO with optional DARA, or a baseline

* `DARA_MODE='preflight'`: before training, the hold-a landscape is evaluated for every cap in `CAP_GRID` at or above the starting cap; the smallest cap with M ≤ `M_HIGH` is adopted (fixed adaptation rule). If none qualifies, the surrogate is kept and the run is marked infeasible.
* `DARA_MODE='online'`: every `DIAG_EVERY` episodes, the on-policy gate (spread-normalised headroom over constant tiers, `DIAG_EPISODES` each) and M at the current surrogate are evaluated. After `PERSIST` consecutive actionable diagnoses (gate passed and M ≥ `M_HIGH`), the cap is relaxed to the smallest larger grid value with M ≤ `M_HIGH`, followed by `WARMUP_UPDATES` critic-only updates.
* `DARA_MODE='k'`: the original proposal. At every diagnosis that passes the gate, the quality curvature is updated with K ← clip(K + η sign(dominant tier − 2)[w_D (D − D*) + w_Q max(0, ΔQ − ΔQ*)], 0, K_max).

M, D and ΔQ come from hold-a rollouts on the fixed diagnostic states and do not depend on the policy; they are cached per surrogate. Every diagnosis is logged to `dara_log.csv`.
* `DARA_MODE='preflight_price'` / `'online_price'` (second knob): the same diagnosis and cap step as `preflight` / `online`; when DARA acts it then also sets the freeze price ρ (`REBUF_SCALE`, multiplying the training reward's rebuffering penalty) to the value in `PRICE_GRID` whose hold-a landscape gives the lowest intended-return regret of the reward-preferred actions (ties → smaller ρ). No freeze target is needed. If the cap step is infeasible but the price step brings M ≤ `M_HIGH`, the run is reported as fixed by price.

* `DARA_MODE='preflight_lag'` / `'online_lag'` (DARA-L): the same gate, diagnosis and persistence as `preflight` / `online`; when DARA acts it switches on PPO-Lagrangian (`LAGRANGE=1`) with the freeze budget `LAG_TARGET` = the mean freeze per chunk of the intended objective's best hold action over the diagnostic states. Cap, price and quality are not changed, so DARA-L differs from the hand-tuned PPO-Lagrangian baseline only in where the budget comes from.


In [ ]:
def surrogate_cfg(cfg, rn_cap, k, rho=None):
    c = copy.copy(cfg)
    c.RN_CAP, c.QUALITY_K = float(rn_cap), float(k)
    c.REBUF_SCALE = float(cfg.REBUF_SCALE if rho is None else rho)
    c.QUALITY = (list(4.0 * (1.0 - np.exp(-k * np.array(cfg.TIER_MULT)))) if k > 0 else list(cfg.QUALITY_BASE))
    return c

def apply_surrogate(cfg, rn_cap, k, rho=None):
    t = surrogate_cfg(cfg, rn_cap, k, rho)
    cfg.RN_CAP, cfg.QUALITY_K, cfg.QUALITY, cfg.REBUF_SCALE = t.RN_CAP, t.QUALITY_K, t.QUALITY, t.REBUF_SCALE

class DaraController:
    def __init__(self, cfg, norm, states, meta):
        self.cfg, self.norm, self.states, self.meta = cfg, norm, states, meta
        self.cache, self.log, self.p = {}, [], 0
        self.n_changes, self.first_change_ep, self.infeasible = 0, np.nan, 0
        self.initial = (float(cfg.RN_CAP), float(cfg.QUALITY_K), float(cfg.REBUF_SCALE))
        self.price_mode = cfg.DARA_MODE.endswith("_price")
        self.lag_mode = cfg.DARA_MODE.endswith("_lag")

    def act_lag(self, sg):
        # DARA-L: switch on the Lagrangian with the budget the intended objective implies
        cfg = self.cfg
        if cfg.LAGRANGE:
            return False, f"Lagrangian already on (budget {cfg.LAG_TARGET:.3f} s/chunk)"
        cfg.LAG_TARGET = float(sg["intended_freeze_per_chunk"])
        cfg.LAGRANGE = 1
        return True, f"Lagrangian on, budget {cfg.LAG_TARGET:.3f} s/chunk"


    def landscape(self, rn_cap, k, rho=None):
        key = (float(rn_cap), round(float(k), 2), round(float(self.cfg.REBUF_SCALE if rho is None else rho), 4))
        if key not in self.cache:
            c = surrogate_cfg(self.cfg, *key)
            cf = counterfactual(self.states, c, None, None, self.cfg.H_MAIN, modes=("hold",))
            self.cache[key] = landscape_signals(cf, self.meta, c, n_boot=50, seed=self.cfg.SEED)
        return self.cache[key]

    def gate(self, model):
        cfg = self.cfg; seed = cfg.EVAL_SEED + 17
        pol, _ = run_episodes(cfg, self.norm, cfg.DIAG_EPISODES, seed, "policy", model=model)
        tiers = [run_episodes(cfg, self.norm, cfg.DIAG_EPISODES, seed, "tier", tier=b)[0].reward.mean()
                 for b in range(cfg.N_BITRATES)]
        return float((pol.reward.mean() - max(tiers)) / (max(tiers) - min(tiers) + 1e-8))

    def label(self, h, sg):
        cfg = self.cfg
        if h < cfg.GATE_MIN: return "OPT_FAILURE"
        if sg["M"] >= cfg.M_HIGH: return "REWARD_INDUCED" if sg["D"] >= cfg.D_HIGH else "MISALIGNED"
        return "ENV_DEGENERATE" if sg["D"] >= cfg.D_HIGH else "HEALTHY"

    def relax_cap(self, include_current):
        cur, k = float(self.cfg.RN_CAP), float(self.cfg.QUALITY_K)
        for c in self.cfg.CAP_GRID:
            if (c > cur or (include_current and c == cur)) and self.landscape(c, k)["M"] <= self.cfg.M_HIGH:
                return c
        return None

    def choose_price(self, rn_cap):
        # second knob: the freeze price with the lowest intended-return regret of the reward-preferred actions
        k = float(self.cfg.QUALITY_K)
        reg = [self.landscape(rn_cap, k, p)["regret"] for p in self.cfg.PRICE_GRID]
        best = self.cfg.PRICE_GRID[int(np.argmin(reg))]
        print("  price search at cap %g: " % rn_cap + ", ".join(f"rho={p:g}: regret={r:.3f}" for p, r in zip(self.cfg.PRICE_GRID, reg)))
        return best

    def act(self, ep, include_current):
        # cap step (fixed adaptation rule), then, in *_price modes, the price step; returns (changed, action text)
        cfg = self.cfg
        c = self.relax_cap(include_current)
        cap_ok = c is not None
        new_cap = c if cap_ok else float(cfg.RN_CAP)
        new_rho = self.choose_price(new_cap) if self.price_mode else float(cfg.REBUF_SCALE)
        parts = []
        if new_cap != float(cfg.RN_CAP): parts.append(f"cap {cfg.RN_CAP:g} -> {new_cap:g}")
        if new_rho != float(cfg.REBUF_SCALE): parts.append(f"rho {cfg.REBUF_SCALE:g} -> {new_rho:g}")
        fixed = cap_ok or (self.price_mode and self.landscape(new_cap, cfg.QUALITY_K, new_rho)["M"] <= cfg.M_HIGH)
        if not fixed:
            self.infeasible = 1
        if not parts:
            return False, ("aligned: keep surrogate" if cap_ok else "infeasible: keep surrogate")
        apply_surrogate(cfg, new_cap, cfg.QUALITY_K, new_rho)
        tag = "" if cap_ok else (" (fixed by price)" if fixed else " (cap infeasible)")
        return True, "; ".join(parts) + tag

    def record(self, ep, h, sg, lab, action):
        self.log.append(dict(episode=ep, gate=h, D=sg["D"], M=sg["M"], dQ=sg["dQ"], label=lab, persist=self.p,
                             regret=sg.get("regret", np.nan), freeze_excess=sg.get("freeze_excess", np.nan),
                             rn_cap=float(self.cfg.RN_CAP), quality_k=float(self.cfg.QUALITY_K),
                             rebuf_scale=float(self.cfg.REBUF_SCALE), lagrange=int(self.cfg.LAGRANGE),
                             lag_target=float(self.cfg.LAG_TARGET),
                             intended_freeze_per_chunk=sg.get("intended_freeze_per_chunk", np.nan), action=action))
        print(f"[DARA ep {ep}] gate={h:+.3f} D={sg['D']:.3f} M={sg['M']:.3f} dQ={sg['dQ']:.3f} -> {lab} | {action}")

    def changed(self, ep):
        self.n_changes += 1
        if np.isnan(self.first_change_ep): self.first_change_ep = ep

    def preflight(self):
        sg = self.landscape(self.cfg.RN_CAP, self.cfg.QUALITY_K)
        if self.lag_mode:
            lab = self.label(0.0, sg)
            if lab in ("REWARD_INDUCED", "MISALIGNED"):
                changed, action = self.act_lag(sg)
                if changed: self.changed(0)
            else:
                action = "aligned: keep surrogate"
            self.record(0, np.nan, sg, lab, action)
            return
        if self.price_mode:
            lab = self.label(0.0, sg)
            if lab in ("REWARD_INDUCED", "MISALIGNED"):
                changed, action = self.act(0, include_current=True)
                if changed: self.changed(0)
            else:
                action = "aligned: keep surrogate"
            self.record(0, np.nan, sg, lab, action)
            return
        c = self.relax_cap(include_current=True)
        if c is None:
            self.infeasible = 1; action = "infeasible: keep surrogate"
        elif c != float(self.cfg.RN_CAP):
            action = f"cap {self.cfg.RN_CAP:g} -> {c:g}"; apply_surrogate(self.cfg, c, self.cfg.QUALITY_K); self.changed(0)
        else:
            action = "aligned: keep surrogate"
        self.record(0, np.nan, sg, self.label(0.0, sg), action)

    def __call__(self, ep, model):
        cfg = self.cfg
        h = self.gate(model)
        sg = self.landscape(cfg.RN_CAP, cfg.QUALITY_K)
        lab = self.label(h, sg)
        if cfg.DARA_MODE == "online_lag":
            self.p = self.p + 1 if lab in ("REWARD_INDUCED", "MISALIGNED") else 0
            action = "none"
            if self.p >= cfg.PERSIST:
                self.p = 0
                changed, action = self.act_lag(sg)
                self.record(ep, h, sg, lab, action)
                if changed: self.changed(ep)
                return changed
            self.record(ep, h, sg, lab, action); return False
        if cfg.DARA_MODE == "online_price":
            self.p = self.p + 1 if lab in ("REWARD_INDUCED", "MISALIGNED") else 0
            action = "none"
            if self.p >= cfg.PERSIST:
                self.p = 0
                changed, action = self.act(ep, include_current=False)
                self.record(ep, h, sg, lab, action)
                if changed: self.changed(ep)
                return changed
            self.record(ep, h, sg, lab, action); return False
        if cfg.DARA_MODE == "online":
            self.p = self.p + 1 if lab in ("REWARD_INDUCED", "MISALIGNED") else 0
            action = "none"
            if self.p >= cfg.PERSIST:
                c = self.relax_cap(include_current=False); self.p = 0
                if c is None:
                    self.infeasible = 1; action = "infeasible: keep surrogate"
                else:
                    action = f"cap {cfg.RN_CAP:g} -> {c:g}"; apply_surrogate(cfg, c, cfg.QUALITY_K)
                    self.record(ep, h, sg, lab, action); self.changed(ep); return True
            self.record(ep, h, sg, lab, action); return False
        # DARA-k (original proposal)
        if lab == "OPT_FAILURE":
            self.record(ep, h, sg, lab, "gate failed: no change"); return False
        E = cfg.W_D * (sg["D"] - cfg.D_TARGET) + cfg.W_Q * max(0.0, sg["dQ"] - cfg.DQ_TARGET)
        sigma = float(np.sign(sg["dominant_action"] - (cfg.N_BITRATES - 1) / 2))
        k_new = round(float(np.clip(cfg.QUALITY_K + cfg.ETA_K * sigma * E, 0.0, cfg.K_MAX)), 2)
        if abs(k_new - cfg.QUALITY_K) < 0.05:
            self.record(ep, h, sg, lab, f"K stays {cfg.QUALITY_K:g} (E={E:+.3f})"); return False
        action = f"K {cfg.QUALITY_K:g} -> {k_new:g} (E={E:+.3f}, sign={sigma:+.0f})"
        apply_surrogate(cfg, cfg.RN_CAP, k_new)
        self.record(ep, h, sg, lab, action); self.changed(ep); return True

DARA = None
if MODE == "run" and DARA_MODE != "none":
    t0 = time.time()
    DARA = DaraController(C, NORM, DIAG, diag_meta)
    if DARA_MODE in ("preflight", "preflight_price", "preflight_lag"):
        DARA.preflight()
    print(f"DARA ({DARA_MODE}) ready in {time.time()-t0:.0f}s | surrogate: cap={C.RN_CAP:g}, K={C.QUALITY_K:g}, rho={C.REBUF_SCALE:g}, "
          f"lagrange={C.LAGRANGE} budget={C.LAG_TARGET:g}")

In [ ]:
if MODE == "run":
    if ALGO in ("pensieve_official", "corel_official"):
        ppo_log, RMS = run_a3c(model, C, NORM)
    else:
        ppo_log, RMS = run_ppo(model, C, NORM, hook=DARA if DARA_MODE in ('online', 'k', 'online_price', 'online_lag') else None)
    pd.DataFrame(ppo_log).to_csv(RUN_DIR / "ppo_log.csv", index=False)
    torch.save(dict(model_state_dict=model.state_dict(), bc_state_dict=bc_snapshot, cfg=vars(C), norm_stats=NORM,
                    return_norm=None if RMS is None else dict(mean=RMS.mean, var=RMS.var, init=RMS.init)),
               RUN_DIR / "checkpoint.pt")
    model.eval()
    print("checkpoint saved:", RUN_DIR / "checkpoint.pt")
    if DARA is not None:
        pd.DataFrame(DARA.log).to_csv(RUN_DIR / "dara_log.csv", index=False)

## 12. Post-training diagnostics and ground truth

Counterfactual signals and the episode-level ground truth are computed for the trained policy under the surrogate in force at the end of training. Per-episode results (`gt_per_episode.csv`) use the shared evaluation seed, so episodes can be paired across runs on the same environment.

In [ ]:
if MODE == "run":
    CF = {}
    for H in H_LIST:
        t0 = time.time(); CF[H] = counterfactual(DIAG, C, NORM, model, H)
        np.savez_compressed(RUN_DIR / f"counterfactual_H{H}.npz", **CF[H])
        print(f"H={H} done in {time.time()-t0:.0f}s")
if MODE == "run":
    SIG = {}
    for H in H_LIST:
        SIG[H] = full_signals(CF[H], diag_meta, C)
        print(f"H={H}: " + " | ".join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}"
                                      for k, v in SIG[H].items() if not k.endswith(("_lo", "_hi"))))
    per_state = diag_meta.copy()
    cf = CF[H_MAIN]
    per_state["a_star_hold"] = cf["Qh"].argmax(1); per_state["a_pi"] = cf["probs"].argmax(1)
    per_state["a_star_follow"] = cf["Qf"].argmax(1); per_state["a_best_ref"] = ref_returns(cf, C).argmax(1)
    for a in range(N_BITRATES):
        per_state[f"Qh_{a}"] = cf["Qh"][:, a]; per_state[f"Rh_{a}"] = ref_returns(cf, C)[:, a]
        per_state[f"rebh_{a}"] = cf["rebh"][:, a]; per_state[f"pi_{a}"] = cf["probs"][:, a]
    per_state.to_csv(RUN_DIR / "diag_per_state.csv", index=False)
if MODE == "run":
    GT_TABLE, GT, GT_RES = ground_truth(C, NORM, model)
    GT_TABLE.to_csv(RUN_DIR / "ground_truth_episodes.csv")
    pd.DataFrame({f"{k}_{m}": v[m].values for k, v in GT_RES.items() for m in ("reward", "ref", "rebuffer")}
                 ).to_csv(RUN_DIR / "gt_per_episode.csv", index_label="episode")
    print(GT_TABLE.round(2)); print({k: v for k, v in GT.items()})
    checks = {"reward_induced": GT["reward_best_tier"] == N_BITRATES - 1 and GT["ref_best_tier"] < N_BITRATES - 1,
              "env_degenerate": GT["reward_best_tier"] == GT["ref_best_tier"]}
    for key, ok in checks.items():
        if REGIME.startswith(key):
            print(f"construction check for {key}: {'OK' if ok else 'NOT AS INTENDED - inspect before using this run'}")
    if EXPECTED_LABEL != "PROBE":
        print(f"by-construction label {EXPECTED_LABEL} vs independent ground truth {GT['gt_label']}: "
              f"{'agree' if EXPECTED_LABEL == GT['gt_label'] else 'DISAGREE - the regime may not be what it was meant to be'}")

## 13. Summary row for this run

In [ ]:
if MODE == "run":
    rows = []
    for H in H_LIST:
        r = dict(run=RUN_NAME, regime=REGIME, seed=SEED, expected=EXPECTED_LABEL, H=H, is_main_H=int(H == H_MAIN),
                 ladder=LADDER, content_cap=CONTENT_CAP, content_slow=CONTENT_SLOW, content_var_scale=CONTENT_VAR_SCALE, units_fix=UNITS_FIX, size_consistent=SIZE_CONSISTENT, dl_model=DL_MODEL, rn_cap=RN_CAP, clip_lo=CLIP_LO,
                 quality_k=QUALITY_K, return_norm_mode=RETURN_NORM_MODE, value_denorm=VALUE_DENORM,
                 bc_epochs=BC_EPOCHS, ppo_episodes=PPO_EPISODES, ppo_lr=PPO_LR, diag_key=DIAG_KEY,
                 final_ev=(ppo_log[-1]["explained_variance"] if ppo_log else np.nan),
                 dara_mode=DARA_MODE, ent_adapt=ENT_ADAPT, lagrange=LAGRANGE,
                 algo=ALGO, reward_mode=REWARD_MODE,
                 final_mu=(ppo_log[-1].get("mu", np.nan) if ppo_log else np.nan), corel_target=(COREL_TARGET if ALGO == "corel" else np.nan),
                 net_source=NET_SOURCE, net_scenarios=json.dumps(NET_SCENARIOS), net_scale=NET_SCALE,
                 trace_prefix=TRACE_PREFIX, base_chunk_mb=json.dumps(BASE_CHUNK_MB),
                 initial_rn_cap=RN_CAP, initial_quality_k=QUALITY_K,
                 final_rn_cap=float(C.RN_CAP), final_quality_k=float(C.QUALITY_K),
                 final_quality=json.dumps([round(float(x), 3) for x in C.QUALITY]),
                 initial_rebuf_scale=REBUF_SCALE, final_rebuf_scale=float(C.REBUF_SCALE),
                 initial_lagrange=LAGRANGE, final_lagrange=int(C.LAGRANGE), final_lag_target=float(C.LAG_TARGET),
                 dara_changes=(DARA.n_changes if DARA else 0), dara_first_change_ep=(DARA.first_change_ep if DARA else np.nan),
                 dara_infeasible=(DARA.infeasible if DARA else 0),
                 final_c2=(ppo_log[-1]["c2"] if ppo_log else np.nan), final_lambda=(ppo_log[-1]["lam"] if ppo_log else np.nan),
                 J0=GT["policy_ref"])
        r.update(SIG[H]); r.update(GT); rows.append(r)
    SUMMARY = pd.DataFrame(rows)
    SUMMARY.to_csv(RUN_DIR / "triage_summary.csv", index=False)
    print(SUMMARY[SUMMARY.is_main_H == 1][["regime", "expected", "triage", "gt_label", "A", "D", "M", "dQ",
                                           "SCAS_pi", "SCAS_Q", "entropy", "triage_stability"]].to_string(index=False))

## 14. Aggregate (`MODE='aggregate'`, not used for the paper)

Reads every `OUT_ROOT/*/triage_summary.csv` and writes confusion matrices (triage vs by-construction label, triage vs independent ground truth), per-regime signal means, and two figures. `PROBE` runs are summarised but excluded from the confusion matrices.

In [ ]:
LABELS = ["OPT_FAILURE", "ENV_DEGENERATE", "REWARD_INDUCED", "HEALTHY"]
if MODE == "aggregate":
    files = sorted(glob.glob(str(Path(OUT_ROOT) / "*" / "triage_summary.csv")))
    ALL = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    ALL.to_csv(Path(OUT_ROOT) / "aggregate_all_runs.csv", index=False)
    main = ALL[ALL.is_main_H == 1]
    labelled = main[main.expected.isin(LABELS)]
    cm_exp = pd.crosstab(labelled.expected, labelled.triage).reindex(index=LABELS, columns=LABELS, fill_value=0)
    cm_gt = pd.crosstab(main.gt_label, main.triage).reindex(index=LABELS, columns=LABELS, fill_value=0)
    cm_exp.to_csv(Path(OUT_ROOT) / "aggregate_confusion_vs_expected.csv")
    cm_gt.to_csv(Path(OUT_ROOT) / "aggregate_confusion_vs_groundtruth.csv")
    sig_cols = ["A", "D", "M", "M_rebuf", "cap_sat", "dQ", "SCAS_pi", "SCAS_Q", "entropy", "triage_stability"]
    by_regime = main.groupby(["regime", "expected"])[sig_cols].agg(["mean", "std"]).round(3)
    by_regime.to_csv(Path(OUT_ROOT) / "aggregate_signals_by_regime.csv")
    h_sens = ALL.pivot_table(index="run", columns="H", values="triage", aggfunc="first")
    h_sens.to_csv(Path(OUT_ROOT) / "aggregate_H_sensitivity.csv")
    print(f"{len(main)} runs ({len(labelled)} with a by-construction label)")
    print("\ntriage vs expected (rows = expected):\n", cm_exp)
    print("\naccuracy vs expected:", round(np.trace(cm_exp.values) / max(cm_exp.values.sum(), 1), 3))
    print("\ntriage vs independent ground truth (rows = ground truth):\n", cm_gt)
    print("\naccuracy vs ground truth:", round(np.trace(cm_gt.values) / max(cm_gt.values.sum(), 1), 3))
    print("\nexpected vs ground truth agreement:",
          round(float(np.mean(labelled.expected.values == labelled.gt_label.values)), 3) if len(labelled) else "n/a")
    print("\n", by_regime)

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    marks = dict(OPT_FAILURE="x", ENV_DEGENERATE="s", REWARD_INDUCED="^", HEALTHY="o", PROBE="*")
    for lab, g in main.groupby("expected"):
        ax.scatter(g.D, g.M, marker=marks.get(lab, "."), label=lab, s=40)
    ax.axvline(D_HIGH, ls="--", lw=0.8, c="k"); ax.axhline(M_HIGH, ls="--", lw=0.8, c="k")
    ax.set_xlabel("Action dominance D"); ax.set_ylabel("Outcome misalignment M"); ax.legend(fontsize=7, frameon=False)
    fig.tight_layout(); fig.savefig(Path(OUT_ROOT) / "aggregate_D_vs_M.png", dpi=200)

    fig, ax = plt.subplots(figsize=(5, 4.2))
    ax.imshow(cm_exp.values, cmap="Blues")
    ax.set_xticks(range(4)); ax.set_xticklabels([l.replace("_", "\n") for l in LABELS], fontsize=7)
    ax.set_yticks(range(4)); ax.set_yticklabels([l.replace("_", "\n") for l in LABELS], fontsize=7)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, int(cm_exp.values[i, j]), ha="center", va="center", fontsize=9)
    ax.set_xlabel("Triage label"); ax.set_ylabel("By-construction label")
    fig.tight_layout(); fig.savefig(Path(OUT_ROOT) / "aggregate_confusion.png", dpi=200)
    print("aggregate outputs written to", OUT_ROOT)

## 15. Generalisation test (`MODE='generalize'`)

The checkpoint in `GEN_CKPT` is played, without any retraining, on every network × content condition of
`gen_conditions()` (`GEN_NETWORKS` × `GEN_CONTENT`): the training generator, real mmWave traces (Lumos5G) and real
commercial 5G traces (Irish 5G, Raca et al., MMSys 2020), each with the ×1, ×4 and ×8 chunk-size ladders. The
`seen / x4` condition reproduces the training-environment test sessions. The reference job (`GEN_CKPT=''`) plays the
always-one-tier policies and RobustMPC on the same sessions.

An optional runtime guard (`RT_MODES`: runtime freeze price and a familiarity check with RobustMPC fallback) is kept
for comparison; the paper evaluates bare policies only (`RT_MODES=["none"]`).


In [ ]:
# ── generalisation test, optional runtime guard (MODE='generalize', no training) ─────────────────────────────
# A trained checkpoint is evaluated, unchanged, on networks and chunk sizes it never saw. Each condition is
# played with the bare policy and with the runtime guard variants:
#   price    : runtime freeze price. score(a) = ln pi(a|s) - lambda * F(a), where F(a) is the freeze the player
#              expects for tier a from a cautious throughput estimate (RobustMPC's harmonic mean / (1 + max recent
#              error)); lambda <- clip(lambda + RT_ETA * (mean freeze over the last RT_WINDOW chunks - budget), 0, RT_LAMBDA_MAX)
#   fallback : familiarity check. novelty = mean |z| of the policy's normalised inputs; above the RT_OOD_QUANT
#              quantile of the training condition for RT_OOD_PATIENCE chunks in a row -> RobustMPC decides, until
#              RT_OOD_PATIENCE familiar chunks in a row
#   both     : price and fallback together
# The reference job (GEN_CKPT = "" or "none") plays the always-one-tier policies and RobustMPC on the same episodes.

def gen_conditions():
    nets = {
        "seen":          dict(NET_SOURCE="generator", NET_SCENARIOS=[], NET_SCALE=1.0),
        "gen_harsh":     dict(NET_SOURCE="generator", NET_SCENARIOS=["urban_canyon", "tunnel", "under_bridge", "crowded_stationary"], NET_SCALE=1.0),
        "gen_half":      dict(NET_SOURCE="generator", NET_SCENARIOS=[], NET_SCALE=0.5),
        "irish_static":  dict(NET_SOURCE="trace", TRACE_FILE=IRISH_TRACE_FILE, TRACE_PREFIX="Static", NET_SCALE=1.0),
        "irish_driving": dict(NET_SOURCE="trace", TRACE_FILE=IRISH_TRACE_FILE, TRACE_PREFIX="Driving", NET_SCALE=1.0),
        "lumos5g":         dict(NET_SOURCE="trace", TRACE_FILE=LUMOS_TRACE_FILE, TRACE_PREFIX="", NET_SCALE=1.0),
        "lumos5g_walking": dict(NET_SOURCE="trace", TRACE_FILE=LUMOS_TRACE_FILE, TRACE_PREFIX="Walking", NET_SCALE=1.0),
        "lumos5g_driving": dict(NET_SOURCE="trace", TRACE_FILE=LUMOS_TRACE_FILE, TRACE_PREFIX="Driving", NET_SCALE=1.0),
    }
    # one generator scenario per condition ("scn_<scenario>"), and three groups like COREL's walking / driving /
    # dead-zone split ("grp_<group>"; scenarios drawn with their usual relative weights inside the group)
    nets.update({f"scn_{n}": dict(NET_SOURCE="generator", NET_SCENARIOS=[n], NET_SCALE=1.0) for n in SCENARIO_MIX})
    groups = {"walking": ["urban_pedestrian", "crowded_stationary"],
              "driving": ["urban_driving", "motorway", "suburban_open"],
              "deadzone": ["tunnel", "under_bridge", "urban_canyon"]}
    nets.update({f"grp_{g}": dict(NET_SOURCE="generator", NET_SCENARIOS=v, NET_SCALE=1.0) for g, v in groups.items()})
    base = [1.5, 3.0, 6.0, 12.0, 22.0]
    cont = {
        "x4":        dict(BASE_CHUNK_MB=[4 * b for b in base], CONTENT_VAR_SCALE=1.0),
        "x1":        dict(BASE_CHUNK_MB=list(base), CONTENT_VAR_SCALE=1.0),
        "x8":        dict(BASE_CHUNK_MB=[8 * b for b in base], CONTENT_VAR_SCALE=1.0),
        "x4_bursty": dict(BASE_CHUNK_MB=[4 * b for b in base], CONTENT_VAR_SCALE=1.5),
    }
    return [(n, c, {**nets[n], **cont[c]}) for n in GEN_NETWORKS for c in GEN_CONTENT]

class RuntimeGuard:
    def __init__(self, cfg, model, mode, budget, ood_thr):
        self.cfg, self.model, self.mode, self.budget, self.thr = cfg, model, mode, float(budget), ood_thr
        self.mpc = MPCPolicy(cfg)
        self.reset()
    def reset(self):
        self.lam, self.freezes, self.ood_run, self.ok_run, self.fallback = 0.0, [], 0, 0, False
        self.n = self.n_fb = 0; self.lams = []; self.mpc.reset()
    @staticmethod
    def novelty(obs):
        z = np.concatenate([np.abs(obs["next_sizes"]), np.abs(obs["dl_times"]), np.abs(obs["throughputs"]), [abs(obs["remain"])]])
        return float(z.mean())
    def act(self, obs, env):
        a_mpc = self.mpc.act(obs, env)           # always called, so RobustMPC's error history stays current
        p = policy_probs([obs], self.model)[0]
        self.n += 1
        if self.mode in ("fallback", "both"):
            if self.novelty(obs) > self.thr:
                self.ood_run += 1; self.ok_run = 0
            else:
                self.ok_run += 1; self.ood_run = 0
            if not self.fallback and self.ood_run >= self.cfg.RT_OOD_PATIENCE: self.fallback = True
            elif self.fallback and self.ok_run >= self.cfg.RT_OOD_PATIENCE: self.fallback = False
            if self.fallback:
                self.n_fb += 1; self.lams.append(self.lam)
                return int(a_mpc)
        if self.mode in ("price", "both"):
            recent = [t for t in env.tp_hist[-5:] if t > 0]
            hbw = len(recent) / sum(1.0 / t for t in recent) if recent else 1.0
            err = max(self.mpc.past_errors[-5:]) if self.mpc.past_errors else 0.0
            cbw = max(hbw / (1.0 + err), 1e-6)
            sizes = np.asarray(env._obs_sizes, dtype=float)
            risk = np.maximum(0.0, sizes * env.unit / cbw - env.buffer)
            self.lams.append(self.lam)
            return int(np.argmax(np.log(p + 1e-9) - self.lam * risk))
        self.lams.append(self.lam)
        return int(p.argmax())
    def observe(self, rebuffer):
        self.freezes.append(float(rebuffer))
        if self.mode in ("price", "both"):
            w = self.freezes[-self.cfg.RT_WINDOW:]
            self.lam = float(np.clip(self.lam + self.cfg.RT_ETA * (np.mean(w) - self.budget), 0.0, self.cfg.RT_LAMBDA_MAX))

def gen_play(cfg, norm, n, seed, kind, model=None, tier=None, guard=None):
    # mirrors run_episodes (same episode sequence for every method), plus per-episode detail
    env = ABREnv(cfg, norm, "test", seed=seed); mpc = MPCPolicy(cfg) if kind == "mpc" else None
    rows = []; obs = env._state()
    for ep in range(n):
        if ep > 0: obs = env.reset()
        if mpc: mpc.reset()
        if guard: guard.reset()
        done, tiers, sizes = False, [], []
        while not done:
            if kind == "tier": a = tier
            elif kind == "mpc": a = mpc.act(obs, env)
            elif guard is not None: a = guard.act(obs, env)
            else: a = int(policy_probs([obs], model)[0].argmax())
            tiers.append(a); sizes.append(float(np.asarray(env._obs_sizes, dtype=float)[a]))
            obs, _, done, info = env.step(a)
            if guard: guard.observe(info["rebuffer"])
        rows.append(dict(episode=ep, J0=env.ep_ref, reward=env.ep_reward, rebuffer=env.ep_rebuffer, chunks=env.n_chunks,
                         mean_tier=float(np.mean(tiers)), switches=int(np.sum(np.diff(tiers) != 0)),
                         scenario=env.scenario, mean_quality=float(np.mean([cfg.REF_QUALITY[t_] for t_ in tiers])),
                         mean_bitrate_mbps=float(np.mean(sizes)) * env.unit / env.CHUNK_DUR,
                         fallback_share=(guard.n_fb / max(guard.n, 1)) if guard else np.nan,
                         mean_lambda=float(np.mean(guard.lams)) if guard and guard.lams else np.nan))
    return rows

# training environment of the original runs (content ladder x4, default generator): keeps its rt_budget.json name
_DEFAULT_ENV_TAG = hashlib.md5(json.dumps(["content", [6.0, 12.0, 24.0, 48.0, 88.0], "generator", [], 1.0, ""]).encode()).hexdigest()[:8]

if MODE == "generalize":
    GEN_DIR = Path(OUT_ROOT) / "generalize"; GEN_DIR.mkdir(parents=True, exist_ok=True)
    t_all = time.time()
    C.NET_SOURCE, C.NET_SCENARIOS, C.NET_SCALE, C.TRACE_PREFIX = "generator", [], 1.0, ""
    if GEN_CKPT and str(GEN_CKPT).lower() != "none":
        ck = torch.load(Path(GEN_CKPT) / "checkpoint.pt", map_location=DEVICE, weights_only=False)
        tcfg = ck["cfg"]
        for k in ("LADDER", "UNITS_FIX", "DL_MODEL", "BASE_CHUNK_MB", "CONTENT_MODEL", "CONTENT_CAP", "CONTENT_SLOW",
                  "CONTENT_VAR_SCALE", "MAX_BUF", "SIZE_CONSISTENT", "QUALITY", "REF_QUALITY", "REF_ALPHA", "REF_BETA",
                  "ALPHA", "BETA", "RN_CAP", "CLIP_LO", "CLIP_HI", "REBUF_SCALE", "QUALITY_K",
                  "ALGO", "REWARD_MODE", "NET_SOURCE", "NET_SCENARIOS", "NET_SCALE", "TRACE_FILE", "TRACE_PREFIX"):
            if k in tcfg: setattr(C, k, copy.deepcopy(tcfg[k]))
        GEN_NORM = ck["norm_stats"]
        GEN_MODEL = make_model(C).to(DEVICE); GEN_MODEL.load_state_dict(ck["model_state_dict"]); GEN_MODEL.eval()
        TAG = Path(GEN_CKPT).name
        train_mode = dict(dara_mode=tcfg.get("DARA_MODE"), lagrange=tcfg.get("LAGRANGE"), seed=tcfg.get("SEED"))
    else:
        GEN_NORM, GEN_MODEL, TAG, train_mode = None, None, "reference", {}
    print("generalize:", TAG, "| training env:", C.LADDER, C.BASE_CHUNK_MB, "| ", train_mode)

    # freeze budget for the runtime price: the DARA-L rule on the TRAINING environment (same for every checkpoint)
    budget = float(RT_BUDGET)
    guard_needed = any(m != "none" for m in RT_MODES)
    if GEN_MODEL is not None and budget < 0 and guard_needed:
        env_tag = hashlib.md5(json.dumps([C.LADDER, list(C.BASE_CHUNK_MB), C.NET_SOURCE, sorted(C.NET_SCENARIOS),
                                          float(C.NET_SCALE), C.TRACE_PREFIX]).encode()).hexdigest()[:8]
        bfile = GEN_DIR / ("rt_budget.json" if env_tag == _DEFAULT_ENV_TAG else f"rt_budget_{env_tag}.json")
        if bfile.exists():
            budget = json.load(open(bfile))["budget"]
        else:
            states, key = collect_diag_states(C)
            meta = pd.DataFrame([f for _, f in states])
            cf = counterfactual(states, C, None, None, H_MAIN, modes=("hold",))
            budget = landscape_signals(cf, meta, C, n_boot=20, seed=SEED)["intended_freeze_per_chunk"]
            tmp = bfile.with_suffix(f".tmp{os.getpid()}"); json.dump(dict(budget=budget, diag_key=key), open(tmp, "w")); os.replace(tmp, bfile)
    ood_thr = np.inf
    if GEN_MODEL is not None and guard_needed:
        print(f"runtime freeze budget: {budget:.3f} s per chunk")
        # familiarity threshold: quantile of the novelty score on the TRAINING condition, separate episodes
        env = ABREnv(C, GEN_NORM, "eval", seed=GEN_SEED + 1000); scores = []
        obs = env._state()
        for ep in range(30):
            if ep > 0: obs = env.reset()
            done = False
            while not done:
                scores.append(RuntimeGuard.novelty(obs))
                obs, _, done, _ = env.step(int(policy_probs([obs], GEN_MODEL)[0].argmax()))
        ood_thr = float(np.quantile(scores, RT_OOD_QUANT))
        print(f"familiarity threshold (q={RT_OOD_QUANT}): {ood_thr:.3f} from {len(scores)} training-condition decisions")

    rows = []
    for net, cont, ov in gen_conditions():
        t0 = time.time()
        c = copy.copy(C)
        for k, v in ov.items(): setattr(c, k, copy.deepcopy(v))
        if GEN_MODEL is None:
            for b in range(c.N_BITRATES):
                rows += [dict(r, network=net, content=cont, method=f"tier{b}") for r in gen_play(c, None, GEN_EPISODES, GEN_SEED, "tier", tier=b)]
            rows += [dict(r, network=net, content=cont, method="robustmpc") for r in gen_play(c, None, GEN_EPISODES, GEN_SEED, "mpc")]
        else:
            for mode in RT_MODES:
                g = None if mode == "none" else RuntimeGuard(c, GEN_MODEL, mode, budget, ood_thr)
                rr = gen_play(c, GEN_NORM, GEN_EPISODES, GEN_SEED, "policy", model=GEN_MODEL, guard=g)
                rows += [dict(r, network=net, content=cont, method="policy" if mode == "none" else f"policy+{mode}") for r in rr]
        sub = pd.DataFrame([r for r in rows if r["network"] == net and r["content"] == cont])
        print(f"{net:14s} {cont:10s} | " + " | ".join(f"{m}: {g_.J0.mean():8.1f}" for m, g_ in sub.groupby("method", sort=False))
              + f" | {time.time()-t0:.0f}s")
    GEN = pd.DataFrame(rows); GEN.insert(0, "run", TAG)
    for k, v in train_mode.items(): GEN[k] = v
    GEN["rt_budget"] = budget if GEN_MODEL is not None else np.nan
    out = GEN_DIR / f"gen_{TAG}.csv"; GEN.to_csv(out, index=False)
    print(f"generalize: {len(GEN)} episodes in {time.time()-t_all:.0f}s -> {out}")


In [ ]:
# ── session timeline (MODE='timeline', no training) ──────────────────────────────────────────────────────────
# Replays the GT_EPISODES test sessions (split 'test', EVAL_SEED, as in the ground truth of each run) with the trained
# greedy policy and logs every chunk: download start/end on the session clock, quality level, stall, buffer, throughput.
# The per-session J0 must equal the run's gt_per_episode.csv (checked below), so the timeline shows exactly the
# sessions behind the reported numbers.
if MODE == "timeline":
    TL_DIR = Path(OUT_ROOT) / "timeline"; TL_DIR.mkdir(parents=True, exist_ok=True)
    items = [r.strip() for r in str(TL_RUNS).split(",") if r.strip()]
    assert items, "set TL_RUNS"
    SKIP = {"MODE", "RUN_NAME", "OUT_ROOT", "SMOKE", "TL_RUNS", "TL_TAG"}
    rows, traces, checks, ref_cfg = [], [], [], None

    def tl_play(c, norm, model, label, n, record_trace=False):
        # same episode sequence for every method: the network and the video are drawn at reset from the env's own
        # random stream, which actions never touch (COREL's forecast noise has a separate stream)
        env = ABREnv(c, norm, "test", seed=c.EVAL_SEED); mpc = MPCPolicy(c) if model is None else None
        obs = env._state(); out = []
        for ep in range(n):
            if ep > 0: obs = env.reset()
            if mpc: mpc.reset()
            if record_trace:
                traces.extend(dict(episode=ep, scenario=env.scenario, second=s_, mbps=float(v_)) for s_, v_ in enumerate(env.tp_trace))
            done = False
            while not done:
                a = int(policy_probs([obs], model)[0].argmax()) if model is not None else int(mpc.act(obs, env))
                c0, b0, last = float(env.clock), float(env.buffer), int(env.last_br)
                obs, _, done, info = env.step(a)
                out.append(dict(method=label, episode=ep, chunk=env.t - 1, n_chunks=env.n_chunks, scenario=env.scenario,
                                tier=a, prev_tier=last, quality=float(c.REF_QUALITY[a]), dl_start=c0,
                                dl_end=c0 + float(info["dl_time"]), dl_time=float(info["dl_time"]), buffer_before=b0,
                                buffer_after=float(env.buffer), stall=float(info["rebuffer"]),
                                throughput=float(env.tp_hist[-1]), j0_chunk=float(info["ref"])))
        return out

    for it in items:
        if it.lower() == "robustmpc":
            continue
        rd = Path(it) if Path(it).exists() else Path("runs/train") / it
        ck = torch.load(rd / "checkpoint.pt", map_location=DEVICE, weights_only=False)
        c = copy.copy(C)
        for k, v in ck["cfg"].items():
            if k not in SKIP:
                setattr(c, k, copy.deepcopy(v))
        if ref_cfg is None: ref_cfg = c
        model = make_model(c).to(DEVICE); model.load_state_dict(ck["model_state_dict"]); model.eval()
        r = tl_play(c, ck["norm_stats"], model, rd.name, c.GT_EPISODES, record_trace=not traces); rows += r
        gt = rd / "gt_per_episode.csv"
        if gt.exists():
            g = pd.read_csv(gt); d = pd.DataFrame(r).groupby("episode").j0_chunk.sum()
            err = float(np.max(np.abs(d.values[:len(g)] - g.policy_ref.values[:len(d)])))
            checks.append(dict(run=rd.name, max_abs_diff_vs_gt=err)); print(f"{rd.name}: max |J0 - gt_per_episode| = {err:.2e}")
    if any(it.lower() == "robustmpc" for it in items):
        assert ref_cfg is not None, "list at least one checkpoint before robustmpc (it uses that run's environment)"
        rows += tl_play(ref_cfg, None, None, "robustmpc", ref_cfg.GT_EPISODES)
    TL = pd.DataFrame(rows); out = TL_DIR / f"timeline_{TL_TAG}.csv"; TL.to_csv(out, index=False)
    pd.DataFrame(traces).to_csv(TL_DIR / f"timeline_{TL_TAG}_network.csv", index=False)
    pd.DataFrame(checks).to_csv(TL_DIR / f"timeline_{TL_TAG}_check.csv", index=False)
    print(f"timeline: {len(TL)} chunks, {TL.method.nunique()} methods -> {out}")
